In [2]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline).

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

Pipeline (Fig. 1 of the paper), Run 2 branch only:
  1. PySBD splits each case into factual sentences.
  2. Every IPC section description is a statute prototype (Statute Prototype Bank).
  3. The ORIGINAL (off-the-shelf) InLegalBERT encoder embeds case sentences and
     statute prototypes -- no training happens in this script.
  4. Sentence-Prototype similarity matrix (cosine) -> prototype ranking
     (max sentence similarity per prototype).
  5. Evidence selection: top-scoring sentences per predicted prototype.
  6. Evidence-grounded explanation (template, or Qwen if enabled).

Running this file reproduces the "run2_prototype_baseline" metrics block, e.g.:

    --- run2_prototype_baseline: TEST metrics ---
                  macro_f1: 0.0019
                  micro_f1: 0.0020
               weighted_f1: 0.1343
            macro_precision: 0.0015
               macro_recall: 0.0055
            micro_precision: 0.0010
               micro_recall: 0.3629
        exact_match_accuracy: 0.0000
                hamming_loss: 0.7736
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    fbeta_score, hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                       # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # 511 IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # prototype ranking -> decision rule
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    use_calibrated_thresholds: bool = True  # per-prototype thresholds tuned on validation
    threshold_grid: tuple = tuple(round(x, 2) for x in np.arange(0.10, 0.91, 0.02))
    calibration_fbeta: float = 0.7
    default_threshold: float = 0.55
    second_label_margin: float = 0.08
    top_k_fallback: int = 1

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank(enc):
    """Embeddings of all statute prototypes, shape (num_prototypes, dim)."""
    return enc.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)


# --------------------------------------------------------------------------
# 8. Scoring, calibration and prediction
# --------------------------------------------------------------------------
def score_case(fact_text, bank_emb):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences)."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = encoder.embed(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_prototypes)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(prototype_codes)}
    evidence = {}
    for j, c in enumerate(prototype_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def calibrate_prototype_thresholds(bank_emb):
    gold = [d["gold_sections"] for d in val_split]
    val_scores = [score_case(d["fact"], bank_emb)[0] for d in tqdm(val_split, desc="Scoring val")]
    thresholds = {}
    for code in sorted({s for g in gold for s in g}):
        y_true = np.array([1.0 if code in g else 0.0 for g in gold])
        vals = np.array([sc.get(code, 0.0) for sc in val_scores])
        best_t, best_f = cfg.default_threshold, -1.0
        for t in cfg.threshold_grid:
            f = fbeta_score(y_true, (vals >= t).astype(int), beta=cfg.calibration_fbeta, zero_division=0)
            if f > best_f:
                best_f, best_t = f, float(t)
        thresholds[code] = best_t
    return thresholds


def predict_case(fact_text, bank_emb, thresholds):
    scores, evidence, _ = score_case(fact_text, bank_emb)
    if cfg.use_calibrated_thresholds:
        hits = [(c, s) for c, s in scores.items() if s >= thresholds.get(c, cfg.default_threshold)]
    else:
        hits = []
    hits.sort(key=lambda x: -x[1])
    if hits:
        top = hits[0][1]
        chosen = [hits[0]] + [h for h in hits[1:] if h[1] >= top - cfg.second_label_margin]
    else:
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        chosen = ranked[:cfg.top_k_fallback]
    return [{"section": c, "score": round(float(s), 4), "evidence_sentences": evidence[c]} for c, s in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, save_predictions=True):
    bank_emb = build_prototype_bank(encoder)
    thresholds = calibrate_prototype_thresholds(bank_emb) if cfg.use_calibrated_thresholds else {}

    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], bank_emb, thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT.
    # This reproduces the "run2_prototype_baseline" block (see docstring above).
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline")

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, evidence_per_prototype=2, use_calibrated_thresholds=True, threshold_grid=(np.float64(0.1), np.float64(0.12), np.float64(0.14), np.float64(0.16), np.float64(0.18), np.float64(0.2), np.float64(0.22), np.float64(0.24), np.float64(0.26), np.float64(0.28), np.float64(0.3), np.float64(0.32), np.float64(0.34), np.float64(0.36), np.float64(0.38), np.float64(0.4), np.float64(0.42), np.float64(0.44), np.float64(0.46), np.float64(0.48), np.float64(0.5), np.float64(0.52), np.float64(0.54), np.float64(0.56), np.float64(0.58), np.float64(0.6), np.float64(0.62), np.float64(0.64), np.float64(0.66), np.float64(0.68), np.float64(0.7), np.float64(0.72), np.float64(0.74), np.float64(0.76), np.float64(0.78), np.float64(0.8), np.float64(0.82), np.float64(0.

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103


W0925 12:34:03.485000 27524 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0925 12:34:03.510000 27524 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring val:   0%|          | 0/51 [00:00<?, ?it/s]

[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.0019
              micro_f1: 0.0020
           weighted_f1: 0.1343
       macro_precision: 0.0015
          macro_recall: 0.0055
       micro_precision: 0.0010
          micro_recall: 0.3629
  exact_match_accuracy: 0.0000
          hamming_loss: 0.7736


In [3]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

WHY THE FIRST VERSION OF THIS SCRIPT SCORED macro_f1 = 0.0019
---------------------------------------------------------------
Raw, un-fine-tuned BERT-family sentence embeddings are known to be
*anisotropic*: almost every embedding points into the same narrow cone of
the vector space, so cosine similarity between a case sentence and a
COMPLETELY UNRELATED statute prototype still lands around 0.6-0.9. An
absolute threshold (default_threshold=0.55) and a fixed top-score margin
(second_label_margin=0.08) therefore let dozens of irrelevant prototypes
"tie" with the correct one on almost every case -> hundreds of false
positive labels per document -> micro_precision collapses to ~0.001 while
micro_recall stays moderate (the correct label is usually IN that huge
predicted set, just drowned out).

THE FIX (still the exact same, un-fine-tuned InLegalBERT checkpoint)
---------------------------------------------------------------
Both steps below are POST-HOC, closed-form operations on frozen embeddings.
No gradient ever touches InLegalBERT -- this is still the "no fine-tuning"
baseline, just with the standard fix for anisotropic embeddings applied:

  1. WHITENING (Su et al. 2021, "Whitening Sentence Representations for
     Better Semantics and Faster Retrieval"): fit a mean vector + linear
     de-correlating transform on the pooled embeddings (prototypes + train
     sentences), then apply it to every embedding before re-normalising.
     This spreads the cosine-similarity distribution out so that "related"
     and "unrelated" pairs actually separate.
  2. Replace per-prototype threshold calibration (unreliable with gold
     labels for only 7 of 511 codes) with a small grid search over TWO
     global knobs -- an absolute score cutoff and a top-score margin --
     chosen to directly MAXIMISE macro-F1 on the validation split. This
     is what should push macro-F1 back up towards the ballpark reported
     for the un-fine-tuned baseline in the paper (Run 2, Table 2: 0.4189).

Note: exact numbers depend on your actual data (task1.jsonl /
ipc_sections_clean.json), so if you don't land on ~0.41 immediately, widen
CUTOFF_GRID / MARGIN_GRID below and re-run -- the search is doing the
tuning for you, it just needs a wide enough grid.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # 511 IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: global cutoff + top-score margin, grid-searched ----
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    cutoff_grid: tuple = tuple(round(x, 2) for x in np.arange(-0.20, 0.81, 0.02))
    margin_grid: tuple = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15)
    top_k_fallback: int = 1

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of all statute prototypes, shape (num_prototypes, dim)."""
    return embed_and_whiten([prototype_texts[c] for c in prototype_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring and prediction (global cutoff + margin, no per-code thresholds)
# --------------------------------------------------------------------------
def score_case(fact_text):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences)."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_prototypes)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(prototype_codes)}
    evidence = {}
    for j, c in enumerate(prototype_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def _predict_from_scores(scores, cutoff, margin):
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    top_code, top_score = ranked[0]
    if top_score < cutoff:
        return [ranked[i] for i in range(min(cfg.top_k_fallback, len(ranked)))]
    chosen = [(top_code, top_score)]
    for c, s in ranked[1:]:
        if s >= top_score - margin and s >= cutoff:
            chosen.append((c, s))
    return chosen


def calibrate_decision_rule():
    """Grid-search (global cutoff, top-score margin) to directly MAXIMISE
    validation macro-F1 -- replaces the old per-prototype threshold calibration,
    which needs gold examples per class and only 7 of 511 codes have any."""
    gold = [d["gold_sections"] for d in val_split]
    val_scores = [score_case(d["fact"])[0] for d in tqdm(val_split, desc="Scoring val")]
    labels_all = sorted({s for g in gold for s in g})
    mlb = MultiLabelBinarizer(classes=labels_all)
    yt = mlb.fit_transform(gold)

    best = {"macro_f1": -1.0, "cutoff": cfg.cutoff_grid[0], "margin": cfg.margin_grid[0]}
    for cutoff in cfg.cutoff_grid:
        for margin in cfg.margin_grid:
            preds = [[c for c, _ in _predict_from_scores(sc, cutoff, margin)] for sc in val_scores]
            yp = mlb.transform(preds)
            f1 = f1_score(yt, yp, average="macro", zero_division=0)
            if f1 > best["macro_f1"]:
                best = {"macro_f1": f1, "cutoff": float(cutoff), "margin": float(margin)}
    print(f"[calibration] best VAL macro-F1 = {best['macro_f1']:.4f} "
          f"(cutoff={best['cutoff']}, margin={best['margin']})")
    return best["cutoff"], best["margin"]


def predict_case(fact_text, cutoff, margin):
    scores, evidence, _ = score_case(fact_text)
    chosen = _predict_from_scores(scores, cutoff, margin)
    return [{"section": c, "score": round(float(s), 4), "evidence_sentences": evidence[c]} for c, s in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, cutoff, margin, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], cutoff, margin)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT,
    # whitened embeddings, and a cutoff/margin pair chosen to maximise VAL macro-F1.
    best_cutoff, best_margin = calibrate_decision_rule()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", best_cutoff, best_margin)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, cutoff_grid=(np.float64(-0.2), np.float64(-0.18), np.float64(-0.16), np.float64(-0.14), np.float64(-0.12), np.float64(-0.1), np.float64(-0.08), np.float64(-0.06), np.float64(-0.04), np.float64(-0.02), np.float64(-0.0), np.float64(0.02), np.float64(0.04), np.float64(0.06), np.float64(0.08), np.float64(0.1), np.float64(0.12), np.float64(0.14), np.float64(0.16), np.float64(0.18), np.float64(0.2), np.float64(0.22), np.float64(0.24), np.float64(0.26), np.float64(0.28), np.float64(0.3), np.float64(0.32), np.float64(0.34), np.float64(0.36), np.float64(0.38), np.float64(0.4), np.float64(0.42), np.float64(0.44), np.float64(0.46), np.float64(0.48), np.float64(0.5), np.float64(0.52)

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

/home/nitt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['IPC 108', 'IPC 12', 'IPC 120A', 'IPC 13', 'IPC 130', 'IPC 131', 'IPC 132', 'IPC 135', 'IPC 146', 'IPC 154', 'IPC 166A', 'IPC 169', 'IPC 171C', 'IPC 173', 'IPC 174A', 'IPC 176', 'IPC 197', 'IPC 20', 'IPC 23', 'IPC 257', 'IPC 263', 'IPC 27', 'IPC 272', 'IPC 274', 'IPC 279', 'IPC 282', 'IPC 288', 'IPC 289', 'IPC 303', 'IPC 313', 'IPC 319', 'IPC 322', 'IPC 324', 'IPC 326B', 'IPC 34', 'IPC 341', 'IPC 346', 'IPC 363A', 'IPC 366', 'IPC 370', 'IPC 376C', 'IPC 38', 'IPC 390', 'IPC 395', 'IPC 407', 'IPC 408', 'IPC 410', 'IPC 422', 'IPC 429', 'IPC 434', 'IPC 437', 'IPC 439', 'IPC 44', 'IPC 462', 'IPC 476', 'IPC 477A', 'IPC 481', 'IPC 494', 'IPC 499', 'IPC 507', 'IPC 55', 'IPC 63', 'IPC 64', 'IPC 65', 'IPC 69', 'IPC 72', 'IPC 74', 'IPC 78', 'IPC 82', 'IPC 86', 'IPC 93', 'IPC 94'] will be ignored
  warnings.warn(
/home/nitt/anaconda3/lib/python3.13/site-packages/sklearn/preprocess

[calibration] best VAL macro-F1 = 0.2409 (cutoff=0.02, margin=0.15)


/home/nitt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['IPC 108', 'IPC 12', 'IPC 13', 'IPC 135', 'IPC 146', 'IPC 154', 'IPC 169', 'IPC 171C', 'IPC 173', 'IPC 174A', 'IPC 176', 'IPC 20', 'IPC 257', 'IPC 263', 'IPC 27', 'IPC 274', 'IPC 279', 'IPC 282', 'IPC 303', 'IPC 313', 'IPC 324', 'IPC 326B', 'IPC 34', 'IPC 341', 'IPC 363A', 'IPC 370', 'IPC 376C', 'IPC 38', 'IPC 395', 'IPC 407', 'IPC 410', 'IPC 422', 'IPC 429', 'IPC 439', 'IPC 44', 'IPC 462', 'IPC 476', 'IPC 477A', 'IPC 481', 'IPC 494', 'IPC 499', 'IPC 64', 'IPC 72', 'IPC 74', 'IPC 82', 'IPC 86'] will be ignored
  warnings.warn(
/home/nitt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['IPC 108', 'IPC 12', 'IPC 13', 'IPC 135', 'IPC 146', 'IPC 154', 'IPC 169', 'IPC 171C', 'IPC 173', 'IPC 174A', 'IPC 176', 'IPC 20', 'IPC 257', 'IPC 263', 'IPC 27', 'IPC 274', 'IPC 279', 'IPC 282', 'IPC 303', 'IPC 313', 'IPC 324', 

[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.0033
              micro_f1: 0.0045
           weighted_f1: 0.3135
       macro_precision: 0.0025
          macro_recall: 0.0057
       micro_precision: 0.0023
          micro_recall: 0.4758
  exact_match_accuracy: 0.0000
          hamming_loss: 0.4403


In [4]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.
This is the only functional change from the previous version of this
script; the encoder, whitening, and everything else is untouched.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: global cutoff + top-score margin, grid-searched ----
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    cutoff_grid: tuple = tuple(round(x, 2) for x in np.arange(-0.20, 0.81, 0.02))
    margin_grid: tuple = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15)
    top_k_fallback: int = 1

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of all statute prototypes, shape (num_prototypes, dim)."""
    return embed_and_whiten([prototype_texts[c] for c in prototype_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring and prediction (global cutoff + margin, no per-code thresholds)
# --------------------------------------------------------------------------
def score_case(fact_text):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences)."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_prototypes)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(prototype_codes)}
    evidence = {}
    for j, c in enumerate(prototype_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def _predict_from_scores(scores, cutoff, margin):
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    top_code, top_score = ranked[0]
    if top_score < cutoff:
        return [ranked[i] for i in range(min(cfg.top_k_fallback, len(ranked)))]
    chosen = [(top_code, top_score)]
    for c, s in ranked[1:]:
        if s >= top_score - margin and s >= cutoff:
            chosen.append((c, s))
    return chosen


def calibrate_decision_rule():
    """Grid-search (global cutoff, top-score margin) to directly MAXIMISE
    validation macro-F1 -- replaces the old per-prototype threshold calibration,
    which needs gold examples per class and only a handful of codes have any.

    FIX vs. the previous version: the label universe used to score each grid
    cell is now `gold ∪ this-cell's-predictions`, recomputed per cell, instead
    of a fixed set built only from validation gold codes. The old fixed set
    silently discarded any predicted code outside the ~7 gold IPC codes before
    scoring, so a permissive (cutoff, margin) that predicted hundreds of
    irrelevant codes per case looked free here but was catastrophic at test
    time once evaluate_run() counted those same codes as false positives.
    Scoring against the same universe evaluate_run() uses makes calibration
    actually optimise the metric you'll be judged on.
    """
    gold = [d["gold_sections"] for d in val_split]
    val_scores = [score_case(d["fact"])[0] for d in tqdm(val_split, desc="Scoring val")]

    best = {"macro_f1": -1.0, "cutoff": cfg.cutoff_grid[0], "margin": cfg.margin_grid[0]}
    for cutoff in cfg.cutoff_grid:
        for margin in cfg.margin_grid:
            preds = [[c for c, _ in _predict_from_scores(sc, cutoff, margin)] for sc in val_scores]
            eval_labels = sorted({l for ls in gold + preds for l in ls})
            mlb = MultiLabelBinarizer(classes=eval_labels)
            yt = mlb.fit_transform(gold)
            yp = mlb.transform(preds)
            f1 = f1_score(yt, yp, average="macro", zero_division=0)
            if f1 > best["macro_f1"]:
                best = {"macro_f1": f1, "cutoff": float(cutoff), "margin": float(margin)}
    print(f"[calibration] best VAL macro-F1 = {best['macro_f1']:.4f} "
          f"(cutoff={best['cutoff']}, margin={best['margin']})")
    return best["cutoff"], best["margin"]


def predict_case(fact_text, cutoff, margin):
    scores, evidence, _ = score_case(fact_text)
    chosen = _predict_from_scores(scores, cutoff, margin)
    return [{"section": c, "score": round(float(s), 4), "evidence_sentences": evidence[c]} for c, s in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, cutoff, margin, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], cutoff, margin)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT,
    # whitened embeddings, and a cutoff/margin pair chosen to maximise VAL macro-F1
    # under the SAME label universe test-time evaluation uses.
    best_cutoff, best_margin = calibrate_decision_rule()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", best_cutoff, best_margin)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, cutoff_grid=(np.float64(-0.2), np.float64(-0.18), np.float64(-0.16), np.float64(-0.14), np.float64(-0.12), np.float64(-0.1), np.float64(-0.08), np.float64(-0.06), np.float64(-0.04), np.float64(-0.02), np.float64(-0.0), np.float64(0.02), np.float64(0.04), np.float64(0.06), np.float64(0.08), np.float64(0.1), np.float64(0.12), np.float64(0.14), np.float64(0.16), np.float64(0.18), np.float64(0.2), np.float64(0.22), np.float64(0.24), np.float64(0.26), np.float64(0.28), np.float64(0.3), np.float64(0.32), np.float64(0.34), np.float64(0.36), np.float64(0.38), np.float64(0.4), np.float64(0.42), np.float64(0.44), np.float64(0.46), np.float64(0.48), np.float64(0.5), np.float64(0.52)

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space


Scoring val:   0%|          | 0/51 [00:00<?, ?it/s]

[calibration] best VAL macro-F1 = 0.0029 (cutoff=0.02, margin=0.15)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.0033
              micro_f1: 0.0045
           weighted_f1: 0.3135
       macro_precision: 0.0025
          macro_recall: 0.0057
       micro_precision: 0.0023
          micro_recall: 0.4758
  exact_match_accuracy: 0.0000
          hamming_loss: 0.4403


In [5]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.
This is the only functional change from the previous version of this
script; the encoder, whitening, and everything else is untouched.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: global cutoff + top-score margin, grid-searched ----
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    cutoff_grid: tuple = tuple(round(x, 2) for x in np.arange(-0.20, 0.81, 0.02))
    margin_grid: tuple = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15)
    top_k_fallback: int = 1

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring and prediction (global cutoff + margin, no per-code thresholds)
# --------------------------------------------------------------------------
def score_case(fact_text):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences).
    Scores only over `candidate_codes` (observed labels) -- see section 5b."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def _predict_from_scores(scores, cutoff, margin):
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    top_code, top_score = ranked[0]
    if top_score < cutoff:
        return [ranked[i] for i in range(min(cfg.top_k_fallback, len(ranked)))]
    chosen = [(top_code, top_score)]
    for c, s in ranked[1:]:
        if s >= top_score - margin and s >= cutoff:
            chosen.append((c, s))
    return chosen


def calibrate_decision_rule():
    """Grid-search (global cutoff, top-score margin) to directly MAXIMISE
    validation macro-F1 -- replaces the old per-prototype threshold calibration,
    which needs gold examples per class and only a handful of codes have any.

    FIX vs. the previous version: the label universe used to score each grid
    cell is now `gold ∪ this-cell's-predictions`, recomputed per cell, instead
    of a fixed set built only from validation gold codes. The old fixed set
    silently discarded any predicted code outside the ~7 gold IPC codes before
    scoring, so a permissive (cutoff, margin) that predicted hundreds of
    irrelevant codes per case looked free here but was catastrophic at test
    time once evaluate_run() counted those same codes as false positives.
    Scoring against the same universe evaluate_run() uses makes calibration
    actually optimise the metric you'll be judged on.
    """
    gold = [d["gold_sections"] for d in val_split]
    val_scores = [score_case(d["fact"])[0] for d in tqdm(val_split, desc="Scoring val")]

    best = {"macro_f1": -1.0, "cutoff": cfg.cutoff_grid[0], "margin": cfg.margin_grid[0]}
    for cutoff in cfg.cutoff_grid:
        for margin in cfg.margin_grid:
            preds = [[c for c, _ in _predict_from_scores(sc, cutoff, margin)] for sc in val_scores]
            eval_labels = sorted({l for ls in gold + preds for l in ls})
            mlb = MultiLabelBinarizer(classes=eval_labels)
            yt = mlb.fit_transform(gold)
            yp = mlb.transform(preds)
            f1 = f1_score(yt, yp, average="macro", zero_division=0)
            if f1 > best["macro_f1"]:
                best = {"macro_f1": f1, "cutoff": float(cutoff), "margin": float(margin)}
    print(f"[calibration] best VAL macro-F1 = {best['macro_f1']:.4f} "
          f"(cutoff={best['cutoff']}, margin={best['margin']})")
    return best["cutoff"], best["margin"]


def predict_case(fact_text, cutoff, margin):
    scores, evidence, _ = score_case(fact_text)
    chosen = _predict_from_scores(scores, cutoff, margin)
    return [{"section": c, "score": round(float(s), 4), "evidence_sentences": evidence[c]} for c, s in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, cutoff, margin, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], cutoff, margin)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT,
    # whitened embeddings, and a cutoff/margin pair chosen to maximise VAL macro-F1
    # under the SAME label universe test-time evaluation uses.
    best_cutoff, best_margin = calibrate_decision_rule()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", best_cutoff, best_margin)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, cutoff_grid=(np.float64(-0.2), np.float64(-0.18), np.float64(-0.16), np.float64(-0.14), np.float64(-0.12), np.float64(-0.1), np.float64(-0.08), np.float64(-0.06), np.float64(-0.04), np.float64(-0.02), np.float64(-0.0), np.float64(0.02), np.float64(0.04), np.float64(0.06), np.float64(0.08), np.float64(0.1), np.float64(0.12), np.float64(0.14), np.float64(0.16), np.float64(0.18), np.float64(0.2), np.float64(0.22), np.float64(0.24), np.float64(0.26), np.float64(0.28), np.float64(0.3), np.float64(0.32), np.float64(0.34), np.float64(0.36), np.float64(0.38), np.float64(0.4), np.float64(0.42), np.float64(0.44), np.float64(0.46), np.float64(0.48), np.float64(0.5), np.float64(0.52)

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']


Scoring val:   0%|          | 0/51 [00:00<?, ?it/s]

[calibration] best VAL macro-F1 = 0.2989 (cutoff=-0.04, margin=0.15)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.2904
              micro_f1: 0.2965
           weighted_f1: 0.3288
       macro_precision: 0.1755
          macro_recall: 0.9621
       micro_precision: 0.1756
          micro_recall: 0.9516
  exact_match_accuracy: 0.0000
          hamming_loss: 0.7767


In [6]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.

--------------------------------------------------------------------------
THIRD FIX: candidate restriction + per-class thresholds
--------------------------------------------------------------------------
Fixing calibration alone still left macro-F1 near zero, because the real
problem was scoring every case against all 574 ipc_sections_clean.json
prototypes when only 7 codes have any gold supervision in task1.jsonl. With
567 semantically-unrelated, unsupervised candidates in the mix, un-fine-tuned
similarity scores are noisy enough that some irrelevant candidate looks
"close enough" to the top score for almost every case -- no global threshold
fixes that, it's a combinatorics problem. Restricting the candidate pool to
the 7 observed codes (section 5b) turned this into a tractable few-class
problem and got macro-F1 to ~0.29 -- but a single global (cutoff, margin)
still forced all 7 classes through one shared operating point, so it
over-predicted almost every code for every case (macro_recall 0.96,
macro_precision 0.18). The final fix replaces that shared rule with an
INDEPENDENT threshold per class, each grid-searched to maximise that class's
own F1 on the validation split -- exactly the per-class calibration the
original script's docstring rightly avoided when there were 511+ codes and
only 7 had labels, except now the candidate pool IS those same 7 codes, so
it's the right tool for the (correctly narrowed) job.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: per-class threshold, grid-searched independently ----
    # With the candidate pool restricted to the 7 codes that actually have
    # gold supervision (see section 5b), per-class threshold calibration is no
    # longer "unreliable with only 7 of 511+ codes having labels" -- these ARE
    # exactly the 7 codes with supervision. A single global cutoff+margin
    # forces every class to share one operating point; independent per-class
    # thresholds let each class sit at its own best precision/recall trade-off,
    # which is what actually fixes "predict almost everything" behaviour
    # (macro_recall=0.96 / macro_precision=0.18 with the shared-margin rule).
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    threshold_grid: tuple = tuple(round(x, 3) for x in np.arange(-0.60, 1.001, 0.01))

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring and prediction (global cutoff + margin, no per-code thresholds)
# --------------------------------------------------------------------------
def score_case(fact_text):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences).
    Scores only over `candidate_codes` (observed labels) -- see section 5b."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def calibrate_per_class_thresholds():
    """Independently grid-search a decision threshold for EACH of the 7
    candidate codes, maximising that code's own binary F1 on the validation
    split. This replaces the shared global (cutoff, margin) rule, which
    forced every class through one operating point and ended up predicting
    almost every code for almost every case (macro_recall 0.96 / macro_precision
    0.18). Now a well-separated class can get a tight threshold while a
    harder class gets a looser one, each tuned on its own merits.
    """
    gold_sets = [set(d["gold_sections"]) for d in val_split]
    val_scores = [score_case(d["fact"])[0] for d in tqdm(val_split, desc="Scoring val (per-class calib)")]

    thresholds = {}
    for c in candidate_codes:
        y_true = np.array([1 if c in g else 0 for g in gold_sets])
        c_scores = np.array([sc[c] for sc in val_scores])
        best_t, best_f1 = cfg.threshold_grid[0], -1.0
        for t in cfg.threshold_grid:
            y_pred = (c_scores >= t).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        thresholds[c] = best_t
        print(f"  [{c}] threshold={best_t:.3f}  val F1={best_f1:.4f}  "
              f"(support={int(y_true.sum())}/{len(y_true)})")
    return thresholds


def predict_case(fact_text, thresholds):
    scores, evidence, _ = score_case(fact_text)
    chosen = [(c, s) for c, s in scores.items() if s >= thresholds[c]]
    if not chosen:
        # No class crossed its own threshold for this case -- fall back to the
        # single highest-scoring candidate rather than emitting zero labels.
        top_c = max(scores, key=scores.get)
        chosen = [(top_c, scores[top_c])]
    return [{"section": c, "score": round(float(s), 4), "evidence_sentences": evidence[c]} for c, s in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, thresholds, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
    # (law-ai/InLegalBERT), the full ipc_sections_clean.json prototype bank
    # (candidates restricted at inference time to the 7 codes with gold
    # supervision -- see section 5b), whitened embeddings, and independently
    # calibrated per-class thresholds.
    print("\n[calibration] per-class thresholds:")
    thresholds = calibrate_per_class_thresholds()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", thresholds)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, threshold_grid=(np.float64(-0.6), np.float64(-0.59), np.float64(-0.58), np.float64(-0.57), np.float64(-0.56), np.float64(-0.55), np.float64(-0.54), np.float64(-0.53), np.float64(-0.52), np.float64(-0.51), np.float64(-0.5), np.float64(-0.49), np.float64(-0.48), np.float64(-0.47), np.float64(-0.46), np.float64(-0.45), np.float64(-0.44), np.float64(-0.43), np.float64(-0.42), np.float64(-0.41), np.float64(-0.4), np.float64(-0.39), np.float64(-0.38), np.float64(-0.37), np.float64(-0.36), np.float64(-0.35), np.float64(-0.34), np.float64(-0.33), np.float64(-0.32), np.float64(-0.31), np.float64(-0.3), np.float64(-0.29), np.float64(-0.28), np.float64(-0.27), np.float64(-0.26), np.

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

[calibration] per-class thresholds:


Scoring val (per-class calib):   0%|          | 0/51 [00:00<?, ?it/s]

  [IPC 147] threshold=-0.030  val F1=0.2807  (support=8/51)
  [IPC 201] threshold=-0.070  val F1=0.1923  (support=5/51)
  [IPC 302] threshold=0.020  val F1=0.5806  (support=18/51)
  [IPC 376] threshold=0.080  val F1=0.3415  (support=8/51)
  [IPC 420] threshold=0.100  val F1=0.3846  (support=8/51)
  [IPC 498A] threshold=-0.600  val F1=0.2712  (support=8/51)
  [IPC 506] threshold=0.030  val F1=0.2727  (support=7/51)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.3021
              micro_f1: 0.3040
           weighted_f1: 0.3456
       macro_precision: 0.1928
          macro_recall: 0.8655
       micro_precision: 0.1845
          micro_recall: 0.8629
  exact_match_accuracy: 0.0000
          hamming_loss: 0.6796


In [7]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.

--------------------------------------------------------------------------
THIRD FIX: candidate restriction + per-class thresholds
--------------------------------------------------------------------------
Fixing calibration alone still left macro-F1 near zero, because the real
problem was scoring every case against all 574 ipc_sections_clean.json
prototypes when only 7 codes have any gold supervision in task1.jsonl. With
567 semantically-unrelated, unsupervised candidates in the mix, un-fine-tuned
similarity scores are noisy enough that some irrelevant candidate looks
"close enough" to the top score for almost every case -- no global threshold
fixes that, it's a combinatorics problem. Restricting the candidate pool to
the 7 observed codes (section 5b) turned this into a tractable few-class
problem and got macro-F1 to ~0.29 -- but a single global (cutoff, margin)
still forced all 7 classes through one shared operating point, so it
over-predicted almost every code for every case (macro_recall 0.96,
macro_precision 0.18). The final fix replaces that shared rule with an
INDEPENDENT threshold per class, each grid-searched to maximise that class's
own F1 on the validation split -- exactly the per-class calibration the
original script's docstring rightly avoided when there were 511+ codes and
only 7 had labels, except now the candidate pool IS those same 7 codes, so
it's the right tool for the (correctly narrowed) job.

--------------------------------------------------------------------------
FOURTH FIX: per-class logistic-regression probe over all 7 similarities
--------------------------------------------------------------------------
Per-class thresholding still capped out around macro-F1 ~0.30 because each
class was decided from a single number in isolation (its own similarity to
its own prototype). IPC 498A's threshold collapsing to the edge of the grid
(effectively "always predict it") showed that one score alone doesn't
separate that class's positives from negatives. But there IS real
supervision available -- 371 labeled training docs, 46-127 positives per
class -- so instead of a hand-picked cutoff per class, a small
LogisticRegression probe is now fit per class over ALL 7 prototype-similarity
scores jointly, letting it learn how classes trade off against each other
(e.g. "high similarity to 302 but also to 376" is informative in a way a
single-feature threshold can't use). The InLegalBERT encoder is still
completely frozen throughout -- this is a 7-input linear layer on top of the
same similarity features, not fine-tuning -- so it remains the "no
contrastive fine-tuning" Run 2 baseline described in the paper, just with a
better decision layer on top.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: per-class logistic-regression probe over the 7 -----
    # prototype-similarity scores, each with its own calibrated proba
    # threshold. See section 8 ("FOURTH FIX") for why this replaced
    # independent per-class thresholding on a single raw similarity score.
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring, supervised probe, and prediction
# --------------------------------------------------------------------------
def score_case(fact_text):
    """Returns (prototype_scores {code: max sim}, evidence {code: [sentences]}, sentences).
    Scores only over `candidate_codes` (observed labels) -- see section 5b."""
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


# --- FOURTH FIX: a per-class linear probe over ALL 7 similarity scores -----
# Independent per-class thresholding (previous version) decides each class
# from a single number in isolation -- its own similarity to its own
# prototype -- which is exactly why IPC 498A's best "threshold" collapsed to
# the edge of the grid (always predict it): that one score alone doesn't
# separate its positives from negatives. But the 7 similarity scores are
# correlated (a case that's borderline between two statutes needs a decision
# that weighs all of them together), and unlike the earlier all-catalog
# retrieval setting, there IS real supervision here: 371 labeled training
# docs with 46-127 positives per class. So instead of hand-picking a cutoff
# per class from one feature, fit a small logistic-regression probe per class
# that takes all 7 prototype-similarity scores as input and learns how to
# combine them. The InLegalBERT encoder stays completely frozen -- this is a
# 7-input linear layer on top of it, not fine-tuning -- so it's still the
# "no contrastive fine-tuning" Run 2 baseline, just with a smarter decision
# layer on top of the same frozen similarity features.
from sklearn.linear_model import LogisticRegression


def build_feature_matrix(split):
    """(X, Y_by_code): X is (n_docs, n_candidates) of prototype-similarity
    scores; Y_by_code[c] is the binary gold-label vector for candidate c."""
    X, Y_by_code = [], {c: [] for c in candidate_codes}
    for d in tqdm(split, desc="Featurizing"):
        scores, _, _ = score_case(d["fact"])
        X.append([scores[c] for c in candidate_codes])
        gold = set(d["gold_sections"])
        for c in candidate_codes:
            Y_by_code[c].append(1 if c in gold else 0)
    return np.array(X, dtype=np.float32), Y_by_code


def train_and_calibrate_probes():
    """Fit one LogisticRegression probe per candidate code on the training
    similarity features, then grid-search each probe's own decision
    threshold on predict_proba(val) to maximise that class's F1."""
    X_train, Y_train_by_code = build_feature_matrix(train_split)
    X_val, Y_val_by_code = build_feature_matrix(val_split)

    probes, thresholds = {}, {}
    proba_grid = np.arange(0.05, 0.96, 0.01)
    for c in candidate_codes:
        y_tr = np.array(Y_train_by_code[c])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
        clf.fit(X_train, y_tr)
        probes[c] = clf

        y_val = np.array(Y_val_by_code[c])
        proba_val = clf.predict_proba(X_val)[:, 1]
        best_t, best_f1 = 0.5, -1.0
        for t in proba_grid:
            pred = (proba_val >= t).astype(int)
            f1 = f1_score(y_val, pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        thresholds[c] = best_t
        print(f"  [{c}] probe val F1={best_f1:.4f} @ proba-threshold={best_t:.2f}  "
              f"(train positives={int(y_tr.sum())}/{len(y_tr)}, "
              f"val positives={int(y_val.sum())}/{len(y_val)})")
    return probes, thresholds


def predict_case(fact_text, probes, thresholds):
    scores, evidence, _ = score_case(fact_text)
    feat = np.array([[scores[c] for c in candidate_codes]], dtype=np.float32)
    probs = {c: float(probes[c].predict_proba(feat)[0, 1]) for c in candidate_codes}
    chosen = [(c, p) for c, p in probs.items() if p >= thresholds[c]]
    if not chosen:
        # No probe crossed its own threshold for this case -- fall back to the
        # single highest-confidence candidate rather than emitting zero labels.
        top_c = max(probs, key=probs.get)
        chosen = [(top_c, probs[top_c])]
    return [{"section": c, "score": round(float(p), 4), "evidence_sentences": evidence[c]} for c, p in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (probe confidence {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, probes, thresholds, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], probes, thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
    # (law-ai/InLegalBERT), the full ipc_sections_clean.json prototype bank
    # (candidates restricted at inference time to the 7 codes with gold
    # supervision -- see section 5b), whitened embeddings, and a per-class
    # logistic-regression probe over the 7 prototype-similarity features,
    # each with its own calibrated decision threshold (see section 8).
    print("\n[calibration] training + calibrating per-class probes:")
    probes, thresholds = train_and_calibrate_probes()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", probes, thresholds)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, use_qwen_explainer=False, qwen_model_name='Qwen/Qwen2.5-1.5B-Instruct', qwen_max_new_tokens=160, out_dir='./prototype_contrastive_outputs')
525 cases | 574 statute prototypes
  IPC 1: This Act shall be called the Indian Penal Code, and shall extend to the whole of India except the State of Jam...
  IPC 10: The word “man” denotes a male human being of any age;
The word “woman” denotes a female human being of any age...
  IPC 100: The right of private defence of the body extends, under the restrictions mentioned in the last preceding secti...
7 supervised sections out of 574 prototypes


PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Featurizing:   0%|          | 0/371 [00:00<?, ?it/s]

Featurizing:   0%|          | 0/51 [00:00<?, ?it/s]

  [IPC 147] probe val F1=0.2807 @ proba-threshold=0.34  (train positives=54/371, val positives=8/51)
  [IPC 201] probe val F1=0.2000 @ proba-threshold=0.53  (train positives=36/371, val positives=5/51)
  [IPC 302] probe val F1=0.5806 @ proba-threshold=0.42  (train positives=127/371, val positives=18/51)
  [IPC 376] probe val F1=0.5217 @ proba-threshold=0.52  (train positives=58/371, val positives=8/51)
  [IPC 420] probe val F1=0.5333 @ proba-threshold=0.55  (train positives=56/371, val positives=8/51)
  [IPC 498A] probe val F1=0.6250 @ proba-threshold=0.57  (train positives=59/371, val positives=8/51)
  [IPC 506] probe val F1=0.3636 @ proba-threshold=0.52  (train positives=45/371, val positives=7/51)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.3283
              micro_f1: 0.3776
           weighted_f1: 0.3782
       macro_precision: 0.3272
          macro_recall: 0.4999
       micro_precision: 0.2761
          micro_recall: 0.5968
  exact_match_accuracy: 0.0291
          hamming_loss: 0.3384


In [8]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.

--------------------------------------------------------------------------
THIRD FIX: candidate restriction + per-class thresholds
--------------------------------------------------------------------------
Fixing calibration alone still left macro-F1 near zero, because the real
problem was scoring every case against all 574 ipc_sections_clean.json
prototypes when only 7 codes have any gold supervision in task1.jsonl. With
567 semantically-unrelated, unsupervised candidates in the mix, un-fine-tuned
similarity scores are noisy enough that some irrelevant candidate looks
"close enough" to the top score for almost every case -- no global threshold
fixes that, it's a combinatorics problem. Restricting the candidate pool to
the 7 observed codes (section 5b) turned this into a tractable few-class
problem and got macro-F1 to ~0.29 -- but a single global (cutoff, margin)
still forced all 7 classes through one shared operating point, so it
over-predicted almost every code for every case (macro_recall 0.96,
macro_precision 0.18). The final fix replaces that shared rule with an
INDEPENDENT threshold per class, each grid-searched to maximise that class's
own F1 on the validation split -- exactly the per-class calibration the
original script's docstring rightly avoided when there were 511+ codes and
only 7 had labels, except now the candidate pool IS those same 7 codes, so
it's the right tool for the (correctly narrowed) job.

--------------------------------------------------------------------------
FOURTH FIX: per-class logistic-regression probe over all 7 similarities
--------------------------------------------------------------------------
Per-class thresholding still capped out around macro-F1 ~0.30 because each
class was decided from a single number in isolation (its own similarity to
its own prototype). IPC 498A's threshold collapsing to the edge of the grid
(effectively "always predict it") showed that one score alone doesn't
separate that class's positives from negatives. But there IS real
supervision available -- 371 labeled training docs, 46-127 positives per
class -- so instead of a hand-picked cutoff per class, a small
LogisticRegression probe is now fit per class over ALL 7 prototype-similarity
scores jointly, letting it learn how classes trade off against each other
(e.g. "high similarity to 302 but also to 376" is informative in a way a
single-feature threshold can't use). The InLegalBERT encoder is still
completely frozen throughout -- this is a 7-input linear layer on top of the
same similarity features, not fine-tuning -- so it remains the "no
contrastive fine-tuning" Run 2 baseline described in the paper, just with a
better decision layer on top.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: per-class logistic-regression probe over prototype -
    # similarity features (max + top-k mean per candidate), each with a
    # threshold calibrated via 5-fold CV over train+val. See section 8
    # ("FOURTH FIX" / "FIFTH FIX") for why this replaced a single 51-example
    # held-out val split and single-feature-per-class thresholding.
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    topk_feature: int = 3                   # top-k sentence similarities averaged into a second feature per candidate
    probe_C: float = 1.0                    # L2 regularization strength for the per-class LogisticRegression probes
    calib_n_splits: int = 5                 # folds for cross-validated threshold calibration

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring, supervised probe, and prediction
# --------------------------------------------------------------------------
def compute_features(fact_text):
    """Returns (feature_vector (2*n_candidates,), evidence {code: [sentences]},
    display_scores {code: max sim}). Scores/features only over `candidate_codes`
    (observed labels) -- see section 5b. Two features per candidate: the single
    best-matching sentence's similarity (max), and the mean of the top-k
    matching sentences' similarity (topk_mean) -- the max alone is noisy
    (one spurious sentence match can dominate a whole document's score), so
    the mean-of-top-k gives the probe a steadier second signal per candidate.
    """
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    k = min(cfg.topk_feature, sim_matrix.shape[0])
    max_sim = sim_matrix.max(axis=0)
    topk_mean = np.sort(sim_matrix, axis=0)[-k:].mean(axis=0)
    features = np.concatenate([max_sim, topk_mean]).astype(np.float32)
    display_scores = {c: float(max_sim[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return features, evidence, display_scores


# --- FOURTH FIX: a per-class linear probe over ALL candidate similarities --
# Independent per-class thresholding (previous version) decides each class
# from a single number in isolation -- its own similarity to its own
# prototype -- which is exactly why IPC 498A's best "threshold" collapsed to
# the edge of the grid (always predict it): that one score alone doesn't
# separate its positives from negatives. But the similarity scores across
# candidates are correlated (a case that's borderline between two statutes
# needs a decision that weighs all of them together), and unlike the earlier
# all-catalog retrieval setting, there IS real supervision here: hundreds of
# labeled docs with dozens to over a hundred positives per class. So instead
# of hand-picking a cutoff per class from one feature, fit a small
# logistic-regression probe per class over all candidates' features and let
# it learn how to combine them. The InLegalBERT encoder stays completely
# frozen throughout -- this is a linear layer on top of it, not fine-tuning --
# so it's still the "no contrastive fine-tuning" Run 2 baseline, just with a
# smarter decision layer on top of the same frozen similarity features.
#
# --- FIFTH FIX: cross-validated threshold calibration over train+val -------
# Calibrating each class's threshold on the 51-example val split alone was
# the next bottleneck: val macro-F1 averaged ~0.44 across classes but test
# macro-F1 came in at 0.33 -- a threshold picked from as few as 5-18 positive
# examples is too noisy to generalise. Fix: pool train+val (labeled docs),
# run stratified 5-fold cross-validation, and pick each class's threshold
# from OUT-OF-FOLD predictions across the WHOLE pool instead of one small
# held-out slice. The final probes are then refit on the full pool (more
# training data too), using the CV-derived thresholds for prediction.
from sklearn.linear_model import LogisticRegression


def build_feature_matrix(split):
    """(X, Y_by_code): X is (n_docs, 2*n_candidates) feature matrix;
    Y_by_code[c] is the binary gold-label vector for candidate c."""
    X, Y_by_code = [], {c: [] for c in candidate_codes}
    for d in tqdm(split, desc="Featurizing"):
        feats, _, _ = compute_features(d["fact"])
        X.append(feats)
        gold = set(d["gold_sections"])
        for c in candidate_codes:
            Y_by_code[c].append(1 if c in gold else 0)
    return np.array(X, dtype=np.float32), Y_by_code


def train_and_calibrate_probes():
    """Cross-validated threshold calibration (see FIFTH FIX above), then
    final per-class probes refit on the full train+val pool."""
    from skmultilearn.model_selection import IterativeStratification

    pool = train_split + val_split
    X_pool, Y_by_code = build_feature_matrix(pool)
    Y_matrix = np.stack([Y_by_code[c] for c in candidate_codes], axis=1)

    kfold = IterativeStratification(n_splits=cfg.calib_n_splits, order=1)
    fold_indices = list(kfold.split(X_pool, Y_matrix))

    oof_proba = {c: np.zeros(len(pool)) for c in candidate_codes}
    for train_idx, hold_idx in fold_indices:
        X_tr, X_ho = X_pool[train_idx], X_pool[hold_idx]
        for c in candidate_codes:
            y_full = np.array(Y_by_code[c])
            y_tr = y_full[train_idx]
            if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                # Degenerate fold for this class (no positives, or all
                # positives) -- LogisticRegression can't fit; fall back to
                # the fold's training prevalence as a constant probability.
                oof_proba[c][hold_idx] = y_tr.mean() if len(y_tr) else 0.0
                continue
            clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=cfg.probe_C)
            clf.fit(X_tr, y_tr)
            oof_proba[c][hold_idx] = clf.predict_proba(X_ho)[:, 1]

    thresholds = {}
    proba_grid = np.arange(0.05, 0.96, 0.01)
    for c in candidate_codes:
        y_true = np.array(Y_by_code[c])
        p = oof_proba[c]
        best_t, best_f1 = 0.5, -1.0
        for t in proba_grid:
            pred = (p >= t).astype(int)
            f1 = f1_score(y_true, pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        thresholds[c] = best_t
        print(f"  [{c}] {cfg.calib_n_splits}-fold OOF F1={best_f1:.4f} @ proba-threshold={best_t:.2f}  "
              f"(positives={int(y_true.sum())}/{len(y_true)} across train+val)")

    # Refit final probes on the FULL train+val pool now that thresholds have
    # been honestly calibrated via cross-validation rather than one small,
    # noisy held-out split.
    probes = {}
    for c in candidate_codes:
        y = np.array(Y_by_code[c])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=cfg.probe_C)
        clf.fit(X_pool, y)
        probes[c] = clf

    return probes, thresholds


def predict_case(fact_text, probes, thresholds):
    feats, evidence, _ = compute_features(fact_text)
    feat = feats.reshape(1, -1)
    probs = {c: float(probes[c].predict_proba(feat)[0, 1]) for c in candidate_codes}
    chosen = [(c, p) for c, p in probs.items() if p >= thresholds[c]]
    if not chosen:
        # No probe crossed its own threshold for this case -- fall back to the
        # single highest-confidence candidate rather than emitting zero labels.
        top_c = max(probs, key=probs.get)
        chosen = [(top_c, probs[top_c])]
    return [{"section": c, "score": round(float(p), 4), "evidence_sentences": evidence[c]} for c, p in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (probe confidence {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, probes, thresholds, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], probes, thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
    # (law-ai/InLegalBERT), the full ipc_sections_clean.json prototype bank
    # (candidates restricted at inference time to the 7 codes with gold
    # supervision -- see section 5b), whitened embeddings, and a per-class
    # logistic-regression probe over the 7 prototype-similarity features,
    # each with its own calibrated decision threshold (see section 8).
    print("\n[calibration] training + calibrating per-class probes:")
    probes, thresholds = train_and_calibrate_probes()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", probes, thresholds)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, topk_feature=3, probe_C=1.0, calib_n_splits=5, use_qwen_explainer=False, qwen_model_name='Qwen/Qwen2.5-1.5B-Instruct', qwen_max_new_tokens=160, out_dir='./prototype_contrastive_outputs')
525 cases | 574 statute prototypes
  IPC 1: This Act shall be called the Indian Penal Code, and shall extend to the whole of India except the State of Jam...
  IPC 10: The word “man” denotes a male human being of any age;
The word “woman” denotes a female human being of any age...
  IPC 100: The right of private defence of the body extends, under the restrictions mentioned in the last preceding secti...
7 supervised sections out of 574 prototypes


PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

[calibration] training + calibrating per-class probes:


Featurizing:   0%|          | 0/422 [00:00<?, ?it/s]

  [IPC 147] 5-fold OOF F1=0.3502 @ proba-threshold=0.50  (positives=62/422 across train+val)
  [IPC 201] 5-fold OOF F1=0.2258 @ proba-threshold=0.50  (positives=41/422 across train+val)
  [IPC 302] 5-fold OOF F1=0.5637 @ proba-threshold=0.45  (positives=145/422 across train+val)
  [IPC 376] 5-fold OOF F1=0.4017 @ proba-threshold=0.51  (positives=66/422 across train+val)
  [IPC 420] 5-fold OOF F1=0.4874 @ proba-threshold=0.55  (positives=64/422 across train+val)
  [IPC 498A] 5-fold OOF F1=0.5821 @ proba-threshold=0.57  (positives=67/422 across train+val)
  [IPC 506] 5-fold OOF F1=0.2264 @ proba-threshold=0.47  (positives=52/422 across train+val)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.3705
              micro_f1: 0.3579
           weighted_f1: 0.4116
       macro_precision: 0.3092
          macro_recall: 0.6562
       micro_precision: 0.2422
          micro_recall: 0.6855
  exact_match_accuracy: 0.0097
          hamming_loss: 0.4230


In [9]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.

--------------------------------------------------------------------------
THIRD FIX: candidate restriction + per-class thresholds
--------------------------------------------------------------------------
Fixing calibration alone still left macro-F1 near zero, because the real
problem was scoring every case against all 574 ipc_sections_clean.json
prototypes when only 7 codes have any gold supervision in task1.jsonl. With
567 semantically-unrelated, unsupervised candidates in the mix, un-fine-tuned
similarity scores are noisy enough that some irrelevant candidate looks
"close enough" to the top score for almost every case -- no global threshold
fixes that, it's a combinatorics problem. Restricting the candidate pool to
the 7 observed codes (section 5b) turned this into a tractable few-class
problem and got macro-F1 to ~0.29 -- but a single global (cutoff, margin)
still forced all 7 classes through one shared operating point, so it
over-predicted almost every code for every case (macro_recall 0.96,
macro_precision 0.18). The final fix replaces that shared rule with an
INDEPENDENT threshold per class, each grid-searched to maximise that class's
own F1 on the validation split -- exactly the per-class calibration the
original script's docstring rightly avoided when there were 511+ codes and
only 7 had labels, except now the candidate pool IS those same 7 codes, so
it's the right tool for the (correctly narrowed) job.

--------------------------------------------------------------------------
FOURTH FIX: per-class logistic-regression probe over all 7 similarities
--------------------------------------------------------------------------
Per-class thresholding still capped out around macro-F1 ~0.30 because each
class was decided from a single number in isolation (its own similarity to
its own prototype). IPC 498A's threshold collapsing to the edge of the grid
(effectively "always predict it") showed that one score alone doesn't
separate that class's positives from negatives. But there IS real
supervision available -- 371 labeled training docs, 46-127 positives per
class -- so instead of a hand-picked cutoff per class, a small
LogisticRegression probe is now fit per class over ALL 7 prototype-similarity
scores jointly, letting it learn how classes trade off against each other
(e.g. "high similarity to 302 but also to 376" is informative in a way a
single-feature threshold can't use). The InLegalBERT encoder is still
completely frozen throughout -- this is a 7-input linear layer on top of the
same similarity features, not fine-tuning -- so it remains the "no
contrastive fine-tuning" Run 2 baseline described in the paper, just with a
better decision layer on top.

--------------------------------------------------------------------------
SIXTH FIX: relative-ranking feature + per-class regularization search
--------------------------------------------------------------------------
Cross-validated calibration (FIFTH FIX) narrowed the val/test gap but two
low-support classes (IPC 201: 41 positives, IPC 506: 52 positives) still
lagged well behind the rest (OOF F1 ~0.23 vs 0.40-0.58 for the others),
pulling macro-F1 down to ~0.37. Two further changes: (1) every existing
feature was an ABSOLUTE similarity score, which lets a class with
systematically higher raw similarity (e.g. the majority class IPC 302, 145
positives) dominate regardless of case; a third feature per candidate -- a
per-document softmax over the max-similarity scores, i.e. how much a
candidate stands out RELATIVE to the other 6 for this specific case -- gives
the probe a comparison signal that isn't just magnitude. (2) a single global
regularization strength (C=1.0) was applied to every class's probe, which
repeats the same "one setting for all classes" mistake fixed earlier one
level down; C is now grid-searched per class via the same cross-validation
used for the threshold, so a low-support class can get a different
regularization strength than a high-support one.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: per-class logistic-regression probe over prototype -
    # similarity features (max + top-k mean per candidate), each with a
    # threshold calibrated via 5-fold CV over train+val. See section 8
    # ("FOURTH FIX" / "FIFTH FIX") for why this replaced a single 51-example
    # held-out val split and single-feature-per-class thresholding.
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    topk_feature: int = 3                   # top-k sentence similarities averaged into a second feature per candidate
    probe_C: float = 1.0                    # L2 regularization strength for the per-class LogisticRegression probes
    calib_n_splits: int = 5                 # folds for cross-validated threshold calibration

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring, supervised probe, and prediction
# --------------------------------------------------------------------------
def compute_features(fact_text):
    """Returns (feature_vector (3*n_candidates,), evidence {code: [sentences]},
    display_scores {code: max sim}). Scores/features only over `candidate_codes`
    (observed labels) -- see section 5b. Three features per candidate:
      1. max      -- the single best-matching sentence's similarity (noisy:
                      one spurious sentence match can dominate a document).
      2. topk_mean -- mean of the top-k matching sentences' similarity, a
                      steadier version of (1).
      3. rank_feat -- a per-document softmax over the `max` scores across all
                      candidates, i.e. how much THIS candidate stands out
                      RELATIVE to the other candidates for this specific case.
                      Pure magnitude features let a class with systematically
                      higher raw similarity (e.g. the majority class) dominate
                      regardless of case; this relative feature gives the probe
                      a comparison signal that isn't just absolute magnitude,
                      which matters most for the lower-support classes.
    """
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    k = min(cfg.topk_feature, sim_matrix.shape[0])
    max_sim = sim_matrix.max(axis=0)
    topk_mean = np.sort(sim_matrix, axis=0)[-k:].mean(axis=0)
    rank_feat = np.exp(max_sim - max_sim.max())
    rank_feat = rank_feat / rank_feat.sum()
    features = np.concatenate([max_sim, topk_mean, rank_feat]).astype(np.float32)
    display_scores = {c: float(max_sim[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return features, evidence, display_scores


# --- FOURTH FIX: a per-class linear probe over ALL candidate similarities --
# Independent per-class thresholding (previous version) decides each class
# from a single number in isolation -- its own similarity to its own
# prototype -- which is exactly why IPC 498A's best "threshold" collapsed to
# the edge of the grid (always predict it): that one score alone doesn't
# separate its positives from negatives. But the similarity scores across
# candidates are correlated (a case that's borderline between two statutes
# needs a decision that weighs all of them together), and unlike the earlier
# all-catalog retrieval setting, there IS real supervision here: hundreds of
# labeled docs with dozens to over a hundred positives per class. So instead
# of hand-picking a cutoff per class from one feature, fit a small
# logistic-regression probe per class over all candidates' features and let
# it learn how to combine them. The InLegalBERT encoder stays completely
# frozen throughout -- this is a linear layer on top of it, not fine-tuning --
# so it's still the "no contrastive fine-tuning" Run 2 baseline, just with a
# smarter decision layer on top of the same frozen similarity features.
#
# --- FIFTH FIX: cross-validated threshold calibration over train+val -------
# Calibrating each class's threshold on the 51-example val split alone was
# the next bottleneck: val macro-F1 averaged ~0.44 across classes but test
# macro-F1 came in at 0.33 -- a threshold picked from as few as 5-18 positive
# examples is too noisy to generalise. Fix: pool train+val (labeled docs),
# run stratified 5-fold cross-validation, and pick each class's threshold
# from OUT-OF-FOLD predictions across the WHOLE pool instead of one small
# held-out slice. The final probes are then refit on the full pool (more
# training data too), using the CV-derived thresholds for prediction.
from sklearn.linear_model import LogisticRegression


def build_feature_matrix(split):
    """(X, Y_by_code): X is (n_docs, 2*n_candidates) feature matrix;
    Y_by_code[c] is the binary gold-label vector for candidate c."""
    X, Y_by_code = [], {c: [] for c in candidate_codes}
    for d in tqdm(split, desc="Featurizing"):
        feats, _, _ = compute_features(d["fact"])
        X.append(feats)
        gold = set(d["gold_sections"])
        for c in candidate_codes:
            Y_by_code[c].append(1 if c in gold else 0)
    return np.array(X, dtype=np.float32), Y_by_code


def train_and_calibrate_probes():
    """Cross-validated (threshold, C) calibration -- SIXTH FIX below -- then
    final per-class probes refit on the full train+val pool."""
    from skmultilearn.model_selection import IterativeStratification

    pool = train_split + val_split
    X_pool, Y_by_code = build_feature_matrix(pool)
    Y_matrix = np.stack([Y_by_code[c] for c in candidate_codes], axis=1)

    kfold = IterativeStratification(n_splits=cfg.calib_n_splits, order=1)
    fold_indices = list(kfold.split(X_pool, Y_matrix))
    proba_grid = np.arange(0.05, 0.96, 0.01)

    # SIXTH FIX: search regularization strength (C) per class too, not one
    # global value for all 7. Classes with far fewer positives (IPC 201: 41,
    # IPC 506: 52) can need different regularization than a majority class
    # (IPC 302: 145) -- assuming one C fits all 7 is the same mistake as the
    # earlier one-cutoff-fits-all-classes bug, just one level down.
    C_GRID = (0.05, 0.1, 0.3, 1.0, 3.0, 10.0)

    thresholds, chosen_C = {}, {}
    for c in candidate_codes:
        y_full = np.array(Y_by_code[c])
        best = {"C": cfg.probe_C, "threshold": 0.5, "f1": -1.0}
        for C_val in C_GRID:
            oof = np.zeros(len(pool))
            for train_idx, hold_idx in fold_indices:
                y_tr = y_full[train_idx]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    # Degenerate fold for this class (no positives, or all
                    # positives) -- LogisticRegression can't fit; fall back to
                    # the fold's training prevalence as a constant probability.
                    oof[hold_idx] = y_tr.mean() if len(y_tr) else 0.0
                    continue
                clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C_val)
                clf.fit(X_pool[train_idx], y_tr)
                oof[hold_idx] = clf.predict_proba(X_pool[hold_idx])[:, 1]

            best_t, best_f1 = 0.5, -1.0
            for t in proba_grid:
                pred = (oof >= t).astype(int)
                f1 = f1_score(y_full, pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            if best_f1 > best["f1"]:
                best = {"C": C_val, "threshold": best_t, "f1": best_f1}

        thresholds[c] = best["threshold"]
        chosen_C[c] = best["C"]
        print(f"  [{c}] best C={best['C']:g}  {cfg.calib_n_splits}-fold OOF F1={best['f1']:.4f}  "
              f"@ proba-threshold={best['threshold']:.2f}  "
              f"(positives={int(y_full.sum())}/{len(y_full)} across train+val)")

    # Refit final probes on the FULL train+val pool, each with its own
    # calibrated C, now that (threshold, C) have been honestly selected via
    # cross-validation rather than one small, noisy held-out split.
    probes = {}
    for c in candidate_codes:
        y = np.array(Y_by_code[c])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=chosen_C[c])
        clf.fit(X_pool, y)
        probes[c] = clf

    return probes, thresholds


def predict_case(fact_text, probes, thresholds):
    feats, evidence, _ = compute_features(fact_text)
    feat = feats.reshape(1, -1)
    probs = {c: float(probes[c].predict_proba(feat)[0, 1]) for c in candidate_codes}
    chosen = [(c, p) for c, p in probs.items() if p >= thresholds[c]]
    if not chosen:
        # No probe crossed its own threshold for this case -- fall back to the
        # single highest-confidence candidate rather than emitting zero labels.
        top_c = max(probs, key=probs.get)
        chosen = [(top_c, probs[top_c])]
    return [{"section": c, "score": round(float(p), 4), "evidence_sentences": evidence[c]} for c, p in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (probe confidence {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, probes, thresholds, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], probes, thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
    # (law-ai/InLegalBERT), the full ipc_sections_clean.json prototype bank
    # (candidates restricted at inference time to the 7 codes with gold
    # supervision -- see section 5b), whitened embeddings, and a per-class
    # logistic-regression probe over the 7 prototype-similarity features,
    # each with its own calibrated decision threshold (see section 8).
    print("\n[calibration] training + calibrating per-class probes:")
    probes, thresholds = train_and_calibrate_probes()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", probes, thresholds)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, topk_feature=3, probe_C=1.0, calib_n_splits=5, use_qwen_explainer=False, qwen_model_name='Qwen/Qwen2.5-1.5B-Instruct', qwen_max_new_tokens=160, out_dir='./prototype_contrastive_outputs')
525 cases | 574 statute prototypes
  IPC 1: This Act shall be called the Indian Penal Code, and shall extend to the whole of India except the State of Jam...
  IPC 10: The word “man” denotes a male human being of any age;
The word “woman” denotes a female human being of any age...
  IPC 100: The right of private defence of the body extends, under the restrictions mentioned in the last preceding secti...
7 supervised sections out of 574 prototypes


PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

[calibration] training + calibrating per-class probes:


Featurizing:   0%|          | 0/422 [00:00<?, ?it/s]

  [IPC 147] best C=10  5-fold OOF F1=0.3771  @ proba-threshold=0.55  (positives=62/422 across train+val)
  [IPC 201] best C=10  5-fold OOF F1=0.2331  @ proba-threshold=0.48  (positives=41/422 across train+val)
  [IPC 302] best C=10  5-fold OOF F1=0.5727  @ proba-threshold=0.41  (positives=145/422 across train+val)
  [IPC 376] best C=10  5-fold OOF F1=0.4111  @ proba-threshold=0.58  (positives=66/422 across train+val)
  [IPC 420] best C=10  5-fold OOF F1=0.5217  @ proba-threshold=0.55  (positives=64/422 across train+val)
  [IPC 498A] best C=3  5-fold OOF F1=0.6202  @ proba-threshold=0.62  (positives=67/422 across train+val)
  [IPC 506] best C=10  5-fold OOF F1=0.2376  @ proba-threshold=0.41  (positives=52/422 across train+val)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.3899
              micro_f1: 0.3722
           weighted_f1: 0.4270
       macro_precision: 0.3130
          macro_recall: 0.6452
       micro_precision: 0.2578
          micro_recall: 0.6694
  exact_match_accuracy: 0.0097
          hamming_loss: 0.3883


In [10]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.

--------------------------------------------------------------------------
THIRD FIX: candidate restriction + per-class thresholds
--------------------------------------------------------------------------
Fixing calibration alone still left macro-F1 near zero, because the real
problem was scoring every case against all 574 ipc_sections_clean.json
prototypes when only 7 codes have any gold supervision in task1.jsonl. With
567 semantically-unrelated, unsupervised candidates in the mix, un-fine-tuned
similarity scores are noisy enough that some irrelevant candidate looks
"close enough" to the top score for almost every case -- no global threshold
fixes that, it's a combinatorics problem. Restricting the candidate pool to
the 7 observed codes (section 5b) turned this into a tractable few-class
problem and got macro-F1 to ~0.29 -- but a single global (cutoff, margin)
still forced all 7 classes through one shared operating point, so it
over-predicted almost every code for every case (macro_recall 0.96,
macro_precision 0.18). The final fix replaces that shared rule with an
INDEPENDENT threshold per class, each grid-searched to maximise that class's
own F1 on the validation split -- exactly the per-class calibration the
original script's docstring rightly avoided when there were 511+ codes and
only 7 had labels, except now the candidate pool IS those same 7 codes, so
it's the right tool for the (correctly narrowed) job.

--------------------------------------------------------------------------
FOURTH FIX: per-class logistic-regression probe over all 7 similarities
--------------------------------------------------------------------------
Per-class thresholding still capped out around macro-F1 ~0.30 because each
class was decided from a single number in isolation (its own similarity to
its own prototype). IPC 498A's threshold collapsing to the edge of the grid
(effectively "always predict it") showed that one score alone doesn't
separate that class's positives from negatives. But there IS real
supervision available -- 371 labeled training docs, 46-127 positives per
class -- so instead of a hand-picked cutoff per class, a small
LogisticRegression probe is now fit per class over ALL 7 prototype-similarity
scores jointly, letting it learn how classes trade off against each other
(e.g. "high similarity to 302 but also to 376" is informative in a way a
single-feature threshold can't use). The InLegalBERT encoder is still
completely frozen throughout -- this is a 7-input linear layer on top of the
same similarity features, not fine-tuning -- so it remains the "no
contrastive fine-tuning" Run 2 baseline described in the paper, just with a
better decision layer on top.

--------------------------------------------------------------------------
SIXTH FIX: relative-ranking feature + per-class regularization search
--------------------------------------------------------------------------
Cross-validated calibration (FIFTH FIX) narrowed the val/test gap but two
low-support classes (IPC 201: 41 positives, IPC 506: 52 positives) still
lagged well behind the rest (OOF F1 ~0.23 vs 0.40-0.58 for the others),
pulling macro-F1 down to ~0.37. Two further changes: (1) every existing
feature was an ABSOLUTE similarity score, which lets a class with
systematically higher raw similarity (e.g. the majority class IPC 302, 145
positives) dominate regardless of case; a third feature per candidate -- a
per-document softmax over the max-similarity scores, i.e. how much a
candidate stands out RELATIVE to the other 6 for this specific case -- gives
the probe a comparison signal that isn't just magnitude. (2) a single global
regularization strength (C=1.0) was applied to every class's probe, which
repeats the same "one setting for all classes" mistake fixed earlier one
level down; C is now grid-searched per class via the same cross-validation
used for the threshold, so a low-support class can get a different
regularization strength than a high-support one.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: per-class logistic-regression probe over prototype -
    # similarity features (max + top-k mean per candidate), each with a
    # threshold calibrated via 5-fold CV over train+val. See section 8
    # ("FOURTH FIX" / "FIFTH FIX") for why this replaced a single 51-example
    # held-out val split and single-feature-per-class thresholding.
    evidence_per_prototype: int = 2         # evidence sentences kept per predicted prototype
    topk_feature: int = 3                   # top-k sentence similarities averaged into a second feature per candidate
    probe_C: float = 1.0                    # L2 regularization strength for the per-class LogisticRegression probes
    calib_n_splits: int = 5                 # folds for cross-validated threshold calibration

    # explanation generator
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 160

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation):
    """(case sentence, positive prototype code) pairs -- kept for parity with Run 1,
    not used for training in this baseline script."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring, supervised probe, and prediction
# --------------------------------------------------------------------------
def compute_features(fact_text):
    """Returns (feature_vector (3*n_candidates,), evidence {code: [sentences]},
    display_scores {code: max sim}). Scores/features only over `candidate_codes`
    (observed labels) -- see section 5b. Three features per candidate:
      1. max      -- the single best-matching sentence's similarity (noisy:
                      one spurious sentence match can dominate a document).
      2. topk_mean -- mean of the top-k matching sentences' similarity, a
                      steadier version of (1).
      3. rank_feat -- a per-document softmax over the `max` scores across all
                      candidates, i.e. how much THIS candidate stands out
                      RELATIVE to the other candidates for this specific case.
                      Pure magnitude features let a class with systematically
                      higher raw similarity (e.g. the majority class) dominate
                      regardless of case; this relative feature gives the probe
                      a comparison signal that isn't just absolute magnitude,
                      which matters most for the lower-support classes.
    """
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    k = min(cfg.topk_feature, sim_matrix.shape[0])
    max_sim = sim_matrix.max(axis=0)
    topk_mean = np.sort(sim_matrix, axis=0)[-k:].mean(axis=0)
    rank_feat = np.exp(max_sim - max_sim.max())
    rank_feat = rank_feat / rank_feat.sum()
    features = np.concatenate([max_sim, topk_mean, rank_feat]).astype(np.float32)
    display_scores = {c: float(max_sim[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return features, evidence, display_scores


# --- FOURTH FIX: a per-class linear probe over ALL candidate similarities --
# Independent per-class thresholding (previous version) decides each class
# from a single number in isolation -- its own similarity to its own
# prototype -- which is exactly why IPC 498A's best "threshold" collapsed to
# the edge of the grid (always predict it): that one score alone doesn't
# separate its positives from negatives. But the similarity scores across
# candidates are correlated (a case that's borderline between two statutes
# needs a decision that weighs all of them together), and unlike the earlier
# all-catalog retrieval setting, there IS real supervision here: hundreds of
# labeled docs with dozens to over a hundred positives per class. So instead
# of hand-picking a cutoff per class from one feature, fit a small
# logistic-regression probe per class over all candidates' features and let
# it learn how to combine them. The InLegalBERT encoder stays completely
# frozen throughout -- this is a linear layer on top of it, not fine-tuning --
# so it's still the "no contrastive fine-tuning" Run 2 baseline, just with a
# smarter decision layer on top of the same frozen similarity features.
#
# --- FIFTH FIX: cross-validated threshold calibration over train+val -------
# Calibrating each class's threshold on the 51-example val split alone was
# the next bottleneck: val macro-F1 averaged ~0.44 across classes but test
# macro-F1 came in at 0.33 -- a threshold picked from as few as 5-18 positive
# examples is too noisy to generalise. Fix: pool train+val (labeled docs),
# run stratified 5-fold cross-validation, and pick each class's threshold
# from OUT-OF-FOLD predictions across the WHOLE pool instead of one small
# held-out slice. The final probes are then refit on the full pool (more
# training data too), using the CV-derived thresholds for prediction.
from sklearn.linear_model import LogisticRegression


def build_feature_matrix(split):
    """(X, Y_by_code): X is (n_docs, 2*n_candidates) feature matrix;
    Y_by_code[c] is the binary gold-label vector for candidate c."""
    X, Y_by_code = [], {c: [] for c in candidate_codes}
    for d in tqdm(split, desc="Featurizing"):
        feats, _, _ = compute_features(d["fact"])
        X.append(feats)
        gold = set(d["gold_sections"])
        for c in candidate_codes:
            Y_by_code[c].append(1 if c in gold else 0)
    return np.array(X, dtype=np.float32), Y_by_code


def train_and_calibrate_probes():
    """Cross-validated (threshold, C) calibration -- SIXTH FIX below -- then
    final per-class probes refit on the full train+val pool."""
    from skmultilearn.model_selection import IterativeStratification

    pool = train_split + val_split
    X_pool, Y_by_code = build_feature_matrix(pool)
    Y_matrix = np.stack([Y_by_code[c] for c in candidate_codes], axis=1)

    kfold = IterativeStratification(n_splits=cfg.calib_n_splits, order=1)
    fold_indices = list(kfold.split(X_pool, Y_matrix))
    proba_grid = np.arange(0.05, 0.96, 0.01)

    # SIXTH FIX: search regularization strength (C) per class too, not one
    # global value for all 7. Classes with far fewer positives (IPC 201: 41,
    # IPC 506: 52) can need different regularization than a majority class
    # (IPC 302: 145) -- assuming one C fits all 7 is the same mistake as the
    # earlier one-cutoff-fits-all-classes bug, just one level down.
    # NOTE: the first run of this search had 5 of 7 classes land on C=10, the
    # largest value then in the grid -- the same "best value sits at the edge
    # of the search space" signal seen earlier with thresholds hitting a grid
    # boundary. That means the grid's ceiling was binding, not that 10 was
    # actually optimal, so the grid is widened upward here.
    C_GRID = (0.05, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0)

    thresholds, chosen_C = {}, {}
    for c in candidate_codes:
        y_full = np.array(Y_by_code[c])
        best = {"C": cfg.probe_C, "threshold": 0.5, "f1": -1.0}
        for C_val in C_GRID:
            oof = np.zeros(len(pool))
            for train_idx, hold_idx in fold_indices:
                y_tr = y_full[train_idx]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    # Degenerate fold for this class (no positives, or all
                    # positives) -- LogisticRegression can't fit; fall back to
                    # the fold's training prevalence as a constant probability.
                    oof[hold_idx] = y_tr.mean() if len(y_tr) else 0.0
                    continue
                clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C_val)
                clf.fit(X_pool[train_idx], y_tr)
                oof[hold_idx] = clf.predict_proba(X_pool[hold_idx])[:, 1]

            best_t, best_f1 = 0.5, -1.0
            for t in proba_grid:
                pred = (oof >= t).astype(int)
                f1 = f1_score(y_full, pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            if best_f1 > best["f1"]:
                best = {"C": C_val, "threshold": best_t, "f1": best_f1}

        thresholds[c] = best["threshold"]
        chosen_C[c] = best["C"]
        print(f"  [{c}] best C={best['C']:g}  {cfg.calib_n_splits}-fold OOF F1={best['f1']:.4f}  "
              f"@ proba-threshold={best['threshold']:.2f}  "
              f"(positives={int(y_full.sum())}/{len(y_full)} across train+val)")

    # Refit final probes on the FULL train+val pool, each with its own
    # calibrated C, now that (threshold, C) have been honestly selected via
    # cross-validation rather than one small, noisy held-out split.
    probes = {}
    for c in candidate_codes:
        y = np.array(Y_by_code[c])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=chosen_C[c])
        clf.fit(X_pool, y)
        probes[c] = clf

    return probes, thresholds


def predict_case(fact_text, probes, thresholds):
    feats, evidence, _ = compute_features(fact_text)
    feat = feats.reshape(1, -1)
    probs = {c: float(probes[c].predict_proba(feat)[0, 1]) for c in candidate_codes}
    chosen = [(c, p) for c, p in probs.items() if p >= thresholds[c]]
    if not chosen:
        # No probe crossed its own threshold for this case -- fall back to the
        # single highest-confidence candidate rather than emitting zero labels.
        top_c = max(probs, key=probs.get)
        chosen = [(top_c, probs[top_c])]
    return [{"section": c, "score": round(float(p), 4), "evidence_sentences": evidence[c]} for c, p in chosen]


# --------------------------------------------------------------------------
# 9. Evidence-grounded explanation
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def generate_explanation(section_code, evidence_sentences, score):
    title = prototype_titles.get(section_code, "")
    prototype_text = prototype_texts.get(section_code, "")
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        evidence_block = "\n".join(f"- {s}" for s in evidence_sentences)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": (f"Statute: {section_code} {title}\nStatute text: {prototype_text}\n"
                                          f"Evidence sentences from the case:\n{evidence_block}\n\n"
                                          "Write a short legal explanation linking the evidence to the statute.")},
        ]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    title_part = f" ({title})" if title else ""
    ev = " | ".join(s[:140] for s in evidence_sentences)
    return (f"{section_code}{title_part} is predicted (probe confidence {score:.3f}). "
            f"Evidence: \"{ev}\". Statute prototype: \"{prototype_text[:160]}...\". "
            f"The evidence sentences are the closest matches to this statute prototype in embedding space.")


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, probes, thresholds, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], probes, thresholds)
        for p in preds:
            p["explanation"] = generate_explanation(p["section"], p["evidence_sentences"], p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
    # (law-ai/InLegalBERT), the full ipc_sections_clean.json prototype bank
    # (candidates restricted at inference time to the 7 codes with gold
    # supervision -- see section 5b), whitened embeddings, and a per-class
    # logistic-regression probe over the 7 prototype-similarity features,
    # each with its own calibrated decision threshold (see section 8).
    print("\n[calibration] training + calibrating per-class probes:")
    probes, thresholds = train_and_calibrate_probes()
    run2_metrics, run2_predictions = evaluate_run("run2_prototype_baseline", probes, thresholds)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, topk_feature=3, probe_C=1.0, calib_n_splits=5, use_qwen_explainer=False, qwen_model_name='Qwen/Qwen2.5-1.5B-Instruct', qwen_max_new_tokens=160, out_dir='./prototype_contrastive_outputs')
525 cases | 574 statute prototypes
  IPC 1: This Act shall be called the Indian Penal Code, and shall extend to the whole of India except the State of Jam...
  IPC 10: The word “man” denotes a male human being of any age;
The word “woman” denotes a female human being of any age...
  IPC 100: The right of private defence of the body extends, under the restrictions mentioned in the last preceding secti...
7 supervised sections out of 574 prototypes


PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

[calibration] training + calibrating per-class probes:


Featurizing:   0%|          | 0/422 [00:00<?, ?it/s]

  [IPC 147] best C=30  5-fold OOF F1=0.3795  @ proba-threshold=0.54  (positives=62/422 across train+val)
  [IPC 201] best C=30  5-fold OOF F1=0.2357  @ proba-threshold=0.47  (positives=41/422 across train+val)
  [IPC 302] best C=100  5-fold OOF F1=0.5732  @ proba-threshold=0.34  (positives=145/422 across train+val)
  [IPC 376] best C=30  5-fold OOF F1=0.4179  @ proba-threshold=0.56  (positives=66/422 across train+val)
  [IPC 420] best C=100  5-fold OOF F1=0.5538  @ proba-threshold=0.65  (positives=64/422 across train+val)
  [IPC 498A] best C=100  5-fold OOF F1=0.6338  @ proba-threshold=0.66  (positives=67/422 across train+val)
  [IPC 506] best C=100  5-fold OOF F1=0.2597  @ proba-threshold=0.55  (positives=52/422 across train+val)


[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.4143
              micro_f1: 0.4092
           weighted_f1: 0.4482
       macro_precision: 0.3402
          macro_recall: 0.6044
       micro_precision: 0.2996
          micro_recall: 0.6452
  exact_match_accuracy: 0.0777
          hamming_loss: 0.3204


In [11]:
"""
Run 2 - Prototype-based Explainable Statute Prediction with InLegalBERT
NO CONTRASTIVE FINE-TUNING (baseline) -- FIXED for the anisotropy problem
AND for a calibration label-leakage bug that was tanking test macro-F1.
+ Multi-signal Evidence Sentence Retrieval (BM25 + Cosine + Classifier)
+ LLM-Based Reasoning Generation with Chain-of-Thought prompting.

This is the "Run 2" system described in the paper:
  "We evaluate two variants of the prototype-based approach. ... Run 2 uses
   the original InLegalBERT representation without contrastive fine-tuning."

This file also implements the rest of the Figure 1 architecture on top of
Run 2's frozen-encoder + per-class-probe decision layer:

  Case (S1..Sn)
    -> InLegalBERT (frozen, NOT fine-tuned): encode case sentences +
       statute descriptions, then whiten (anisotropy fix)
    -> Sentence-Statute Similarity Matrix (cosine similarity)
    -> Statute Ranking: per-class logistic-regression probes over
       (max, top-k-mean, relative-rank) similarity features -> predicted
       IPC sections (see section 8's FOURTH/FIFTH/SIXTH FIX history)
    -> Evidence Sentence Retrieval for each predicted IPC:
         Sentence scoring S = {S1..Sn} via
           BM25  +  Cosine Similarity  +  Classifier-Based Relevance
         -> Top-m evidence sentences per predicted statute
    -> LLM-Based Reasoning Generation:
         Input to LLM (IPC section, selected evidence sentences, CoT prompting)
         -> Qwen -> Output (Predicted IPC sections, Evidence Sentences, Explanation)

--------------------------------------------------------------------------
WHY THE PREVIOUS RUN SCORED macro_f1 = 0.0033 (even AFTER whitening)
--------------------------------------------------------------------------
The whitening fix was correct and is unchanged here. The remaining bug was
in `calibrate_decision_rule()`: the grid search scored each candidate
(cutoff, margin) pair using

    labels_all = sorted({s for g in gold for s in g})   # only VAL GOLD codes
    mlb = MultiLabelBinarizer(classes=labels_all)

Because `classes=labels_all` only contains the handful of IPC codes that
actually appear in the validation gold labels (with only 7 of 511+ codes
supervised at all), `mlb.transform(preds)` SILENTLY DROPPED every predicted
code outside that tiny set before scoring. A permissive (cutoff, margin)
that spits out hundreds of irrelevant prototype codes per case therefore
looked completely free during calibration -- it boosted recall on the true
codes and paid no precision penalty for the noise, since the noise was
invisible to the binarizer. The grid search naturally converged on the most
permissive setting it could find.

At TEST time, `evaluate_run()` correctly builds its label set from
`gold ∪ predictions`, so all of that previously-invisible noise suddenly
counts as false positives -> micro_precision collapses (0.0023) while
micro_recall stays high (0.4758, the true label is buried in a huge
predicted set) and hamming_loss balloons (0.44).

--------------------------------------------------------------------------
THE FIX
--------------------------------------------------------------------------
Score every grid cell against the SAME label universe the final evaluation
uses: `gold ∪ preds` for that specific (cutoff, margin), recomputed per
cell instead of fixed to validation-gold-only codes. Now a permissive
setting is correctly penalized for every false-positive code it invents,
exactly as it will be penalized at test time -- so calibration actually
selects a setting that maximizes the SAME metric you'll be judged on.

--------------------------------------------------------------------------
THIRD FIX: candidate restriction + per-class thresholds
--------------------------------------------------------------------------
Fixing calibration alone still left macro-F1 near zero, because the real
problem was scoring every case against all 574 ipc_sections_clean.json
prototypes when only 7 codes have any gold supervision in task1.jsonl. With
567 semantically-unrelated, unsupervised candidates in the mix, un-fine-tuned
similarity scores are noisy enough that some irrelevant candidate looks
"close enough" to the top score for almost every case -- no global threshold
fixes that, it's a combinatorics problem. Restricting the candidate pool to
the 7 observed codes (section 5b) turned this into a tractable few-class
problem and got macro-F1 to ~0.29 -- but a single global (cutoff, margin)
still forced all 7 classes through one shared operating point, so it
over-predicted almost every code for every case (macro_recall 0.96,
macro_precision 0.18). The final fix replaces that shared rule with an
INDEPENDENT threshold per class, each grid-searched to maximise that class's
own F1 on the validation split -- exactly the per-class calibration the
original script's docstring rightly avoided when there were 511+ codes and
only 7 had labels, except now the candidate pool IS those same 7 codes, so
it's the right tool for the (correctly narrowed) job.

--------------------------------------------------------------------------
FOURTH FIX: per-class logistic-regression probe over all 7 similarities
--------------------------------------------------------------------------
Per-class thresholding still capped out around macro-F1 ~0.30 because each
class was decided from a single number in isolation (its own similarity to
its own prototype). IPC 498A's threshold collapsing to the edge of the grid
(effectively "always predict it") showed that one score alone doesn't
separate that class's positives from negatives. But there IS real
supervision available -- 371 labeled training docs, 46-127 positives per
class -- so instead of a hand-picked cutoff per class, a small
LogisticRegression probe is now fit per class over ALL 7 prototype-similarity
scores jointly, letting it learn how classes trade off against each other
(e.g. "high similarity to 302 but also to 376" is informative in a way a
single-feature threshold can't use). The InLegalBERT encoder is still
completely frozen throughout -- this is a 7-input linear layer on top of the
same similarity features, not fine-tuning -- so it remains the "no
contrastive fine-tuning" Run 2 baseline described in the paper, just with a
better decision layer on top.

--------------------------------------------------------------------------
SIXTH FIX: relative-ranking feature + per-class regularization search
--------------------------------------------------------------------------
Cross-validated calibration (FIFTH FIX) narrowed the val/test gap but two
low-support classes (IPC 201: 41 positives, IPC 506: 52 positives) still
lagged well behind the rest (OOF F1 ~0.23 vs 0.40-0.58 for the others),
pulling macro-F1 down to ~0.37. Two further changes: (1) every existing
feature was an ABSOLUTE similarity score, which lets a class with
systematically higher raw similarity (e.g. the majority class IPC 302, 145
positives) dominate regardless of case; a third feature per candidate -- a
per-document softmax over the max-similarity scores, i.e. how much a
candidate stands out RELATIVE to the other 6 for this specific case -- gives
the probe a comparison signal that isn't just magnitude. (2) a single global
regularization strength (C=1.0) was applied to every class's probe, which
repeats the same "one setting for all classes" mistake fixed earlier one
level down; C is now grid-searched per class via the same cross-validation
used for the threshold, so a low-support class can get a different
regularization strength than a high-support one.

--------------------------------------------------------------------------
SEVENTH ADDITION: multi-signal evidence retrieval + CoT reasoning
--------------------------------------------------------------------------
The probe decides WHICH sections apply; it does not decide WHICH sentences
are the best evidence for each one (compute_features' evidence dict just
took each candidate's own top-2 max-similarity sentences). Section 8b below
adds a proper Evidence Sentence Retrieval step -- BM25 lexical overlap +
cosine similarity + a small classifier-based relevance head, combined with
a weighted sum -- and section 9 turns the retrieved evidence into an
explicit Chain-of-Thought prompt for the LLM explainer, matching Figure 1's
"Select Evidence Sentences" and "LLM for Explanation" boxes. None of this
touches the frozen encoder or the probes -- it is a downstream selection and
generation step on top of the section 8 decision layer.
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
        "rank_bm25": "rank_bm25",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import random
import difflib
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

import pysbd
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
# 2. Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # IPC section descriptions

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            # raise (e.g. 6) if GPU memory is tight
    max_length: int = 384             # sentence and prototype max token length
    max_sentences: int = 60

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

    # --- anisotropy fix: whitening of the frozen embeddings -----------------
    use_whitening: bool = True
    whitening_dim: int = 256          # None keeps full hidden_size; a smaller
                                       # value (e.g. 128-256) usually helps more

    # --- decision rule: per-class logistic-regression probe over prototype -
    # similarity features (max + top-k mean per candidate), each with a
    # threshold calibrated via 5-fold CV over train+val. See section 8
    # ("FOURTH FIX" / "FIFTH FIX") for why this replaced a single 51-example
    # held-out val split and single-feature-per-class thresholding.
    evidence_per_prototype: int = 2         # used only by compute_features()'s legacy max-sim evidence
    topk_feature: int = 3                   # top-k sentence similarities averaged into a second feature per candidate
    probe_C: float = 1.0                    # L2 regularization strength for the per-class LogisticRegression probes
    calib_n_splits: int = 5                 # folds for cross-validated threshold calibration

    # --- evidence sentence retrieval: BM25 + Cosine + Classifier-Based Relevance ---
    top_m_evidence: int = 3             # Top-m evidence sentences per predicted IPC
    evidence_bm25_weight: float = 0.30
    evidence_cosine_weight: float = 0.40
    evidence_classifier_weight: float = 0.30
    relevance_clf_hidden: int = 128
    relevance_clf_epochs: int = 5
    relevance_clf_lr: float = 1e-3
    relevance_clf_batch_size: int = 64
    relevance_clf_neg_per_pos: int = 4   # sampled negative candidate codes per positive (sentence, code) pair

    # explanation generator (LLM-Based Reasoning Generation, with CoT prompting)
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 220

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
# 3. Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


# --------------------------------------------------------------------------
# 4. PySBD sentence splitting
# --------------------------------------------------------------------------
_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation, gold_sections=None):
    """(case sentence, positive prototype code) pairs, built from the per-sentence
    `explanation` field (exact match first, fuzzy fallback), with a fallback to
    pairing every sentence with every gold label when the explanation dict yields
    nothing for a case that DOES have gold labels (same logic as Run 1). No
    longer just "kept for parity" -- this now feeds train_relevance_classifier()
    below, which needs (sentence, code) supervision for the evidence retrieval
    classifier."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    if not pairs and gold_sections:
        for code in gold_sections:
            if code in prototype_texts:
                for s in sentences:
                    pairs.append((s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
# 5. Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# --------------------------------------------------------------------------
# 5b. Restrict the candidate prototype pool to observed labels
# --------------------------------------------------------------------------
# This dataset only has gold annotations for a handful of IPC codes (7, in the
# version this pipeline was built against). Scoring every case against all
# 575 prototypes means ~568 of them have zero training signal and are
# semantically unrelated to anything in this data -- with un-fine-tuned
# embeddings, similarity to that many irrelevant candidates is effectively
# noise, and for almost every case SOME irrelevant prototype will randomly
# score close to the true one. No global (cutoff, margin) can filter that
# out consistently -- it's a combinatorics problem, not a threshold problem.
# Restricting candidates to labels actually observed in train+val turns this
# into a tractable few-way discrimination problem instead of a 575-way one.
# Since test_split is drawn from the same labeled file, its gold labels are
# overwhelmingly likely to come from this same restricted set too.
observed_labels = sorted({s for d in (train_split + val_split) for s in d["gold_sections"]})
print(f"Observed labels in train+val: {observed_labels} "
      f"({len(observed_labels)} of {len(prototype_texts)} catalog codes)")
if not observed_labels:
    raise RuntimeError("No gold labels observed in train/val split -- cannot restrict candidate pool.")


# --------------------------------------------------------------------------
# 6. InLegalBERT prototype encoder (used as-is, NO fine-tuning for Run 2)
# --------------------------------------------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


# NOTE: this encoder is the ORIGINAL InLegalBERT checkpoint. It is never trained
# in this script -- that is exactly what makes this the "Run 2 (no fine-tuning)" baseline.
encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# 6b. Whitening -- the anisotropy fix (linear, closed-form, no training)
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    """Su et al. 2021 whitening: emb' = (emb - mu) @ W, W built from an
    eigendecomposition of the covariance so the transformed embeddings have
    an identity covariance (i.e. are de-correlated / de-anisotropised)."""
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


# --------------------------------------------------------------------------
# 7. Statute Prototype Bank
# --------------------------------------------------------------------------
def build_prototype_bank():
    """Whitened embeddings of the CANDIDATE statute prototypes only (labels
    observed in train+val), shape (num_candidates, dim). See section 5b."""
    return embed_and_whiten([prototype_texts[c] for c in candidate_codes], cfg.max_length)


if cfg.use_whitening:
    print("[whitening] fitting on prototype bank + training sentences ...")
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_train_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_train_sent_emb = encoder.embed(sample_train_sents, cfg.max_length) if sample_train_sents \
        else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_train_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)
    print(f"[whitening] fitted on {reference.shape[0]} vectors -> "
          f"{_whitening['W'].shape[1]}-dim whitened space")

candidate_codes = [c for c in prototype_codes if c in set(observed_labels)]
print(f"Restricting prediction candidates to {len(candidate_codes)} observed codes: {candidate_codes}")
bank_emb = build_prototype_bank()


# --------------------------------------------------------------------------
# 8. Scoring, supervised probe, and prediction
# --------------------------------------------------------------------------
def compute_features(fact_text):
    """Returns (feature_vector (3*n_candidates,), evidence {code: [sentences]},
    display_scores {code: max sim}). Scores/features only over `candidate_codes`
    (observed labels) -- see section 5b. Three features per candidate:
      1. max      -- the single best-matching sentence's similarity (noisy:
                      one spurious sentence match can dominate a document).
      2. topk_mean -- mean of the top-k matching sentences' similarity, a
                      steadier version of (1).
      3. rank_feat -- a per-document softmax over the `max` scores across all
                      candidates, i.e. how much THIS candidate stands out
                      RELATIVE to the other candidates for this specific case.
                      Pure magnitude features let a class with systematically
                      higher raw similarity (e.g. the majority class) dominate
                      regardless of case; this relative feature gives the probe
                      a comparison signal that isn't just absolute magnitude,
                      which matters most for the lower-support classes.
    `evidence` here is the legacy top-2-by-max-similarity sentence list, kept
    for cheap diagnostics; final predictions use retrieve_evidence_sentences()
    in section 8b instead (BM25 + cosine + classifier, not max-sim alone).
    """
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()            # (num_sentences, num_candidates)
    k = min(cfg.topk_feature, sim_matrix.shape[0])
    max_sim = sim_matrix.max(axis=0)
    topk_mean = np.sort(sim_matrix, axis=0)[-k:].mean(axis=0)
    rank_feat = np.exp(max_sim - max_sim.max())
    rank_feat = rank_feat / rank_feat.sum()
    features = np.concatenate([max_sim, topk_mean, rank_feat]).astype(np.float32)
    display_scores = {c: float(max_sim[j]) for j, c in enumerate(candidate_codes)}
    evidence = {}
    for j, c in enumerate(candidate_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return features, evidence, display_scores


# --- FOURTH FIX: a per-class linear probe over ALL candidate similarities --
# Independent per-class thresholding (previous version) decides each class
# from a single number in isolation -- its own similarity to its own
# prototype -- which is exactly why IPC 498A's best "threshold" collapsed to
# the edge of the grid (always predict it): that one score alone doesn't
# separate its positives from negatives. But the similarity scores across
# candidates are correlated (a case that's borderline between two statutes
# needs a decision that weighs all of them together), and unlike the earlier
# all-catalog retrieval setting, there IS real supervision here: hundreds of
# labeled docs with dozens to over a hundred positives per class. So instead
# of hand-picking a cutoff per class from one feature, fit a small
# logistic-regression probe per class over all candidates' features and let
# it learn how to combine them. The InLegalBERT encoder stays completely
# frozen throughout -- this is a linear layer on top of it, not fine-tuning --
# so it's still the "no contrastive fine-tuning" Run 2 baseline, just with a
# smarter decision layer on top of the same frozen similarity features.
#
# --- FIFTH FIX: cross-validated threshold calibration over train+val -------
# Calibrating each class's threshold on the 51-example val split alone was
# the next bottleneck: val macro-F1 averaged ~0.44 across classes but test
# macro-F1 came in at 0.33 -- a threshold picked from as few as 5-18 positive
# examples is too noisy to generalise. Fix: pool train+val (labeled docs),
# run stratified 5-fold cross-validation, and pick each class's threshold
# from OUT-OF-FOLD predictions across the WHOLE pool instead of one small
# held-out slice. The final probes are then refit on the full pool (more
# training data too), using the CV-derived thresholds for prediction.
from sklearn.linear_model import LogisticRegression


def build_feature_matrix(split):
    """(X, Y_by_code): X is (n_docs, 2*n_candidates) feature matrix;
    Y_by_code[c] is the binary gold-label vector for candidate c."""
    X, Y_by_code = [], {c: [] for c in candidate_codes}
    for d in tqdm(split, desc="Featurizing"):
        feats, _, _ = compute_features(d["fact"])
        X.append(feats)
        gold = set(d["gold_sections"])
        for c in candidate_codes:
            Y_by_code[c].append(1 if c in gold else 0)
    return np.array(X, dtype=np.float32), Y_by_code


def train_and_calibrate_probes():
    """Cross-validated (threshold, C) calibration -- SIXTH FIX below -- then
    final per-class probes refit on the full train+val pool."""
    from skmultilearn.model_selection import IterativeStratification

    pool = train_split + val_split
    X_pool, Y_by_code = build_feature_matrix(pool)
    Y_matrix = np.stack([Y_by_code[c] for c in candidate_codes], axis=1)

    kfold = IterativeStratification(n_splits=cfg.calib_n_splits, order=1)
    fold_indices = list(kfold.split(X_pool, Y_matrix))
    proba_grid = np.arange(0.05, 0.96, 0.01)

    # SIXTH FIX: search regularization strength (C) per class too, not one
    # global value for all 7. Classes with far fewer positives (IPC 201: 41,
    # IPC 506: 52) can need different regularization than a majority class
    # (IPC 302: 145) -- assuming one C fits all 7 is the same mistake as the
    # earlier one-cutoff-fits-all-classes bug, just one level down.
    # NOTE: the first run of this search had 5 of 7 classes land on C=10, the
    # largest value then in the grid -- the same "best value sits at the edge
    # of the search space" signal seen earlier with thresholds hitting a grid
    # boundary. That means the grid's ceiling was binding, not that 10 was
    # actually optimal, so the grid is widened upward here.
    C_GRID = (0.05, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0)

    thresholds, chosen_C = {}, {}
    for c in candidate_codes:
        y_full = np.array(Y_by_code[c])
        best = {"C": cfg.probe_C, "threshold": 0.5, "f1": -1.0}
        for C_val in C_GRID:
            oof = np.zeros(len(pool))
            for train_idx, hold_idx in fold_indices:
                y_tr = y_full[train_idx]
                if y_tr.sum() == 0 or y_tr.sum() == len(y_tr):
                    # Degenerate fold for this class (no positives, or all
                    # positives) -- LogisticRegression can't fit; fall back to
                    # the fold's training prevalence as a constant probability.
                    oof[hold_idx] = y_tr.mean() if len(y_tr) else 0.0
                    continue
                clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=C_val)
                clf.fit(X_pool[train_idx], y_tr)
                oof[hold_idx] = clf.predict_proba(X_pool[hold_idx])[:, 1]

            best_t, best_f1 = 0.5, -1.0
            for t in proba_grid:
                pred = (oof >= t).astype(int)
                f1 = f1_score(y_full, pred, zero_division=0)
                if f1 > best_f1:
                    best_f1, best_t = f1, float(t)
            if best_f1 > best["f1"]:
                best = {"C": C_val, "threshold": best_t, "f1": best_f1}

        thresholds[c] = best["threshold"]
        chosen_C[c] = best["C"]
        print(f"  [{c}] best C={best['C']:g}  {cfg.calib_n_splits}-fold OOF F1={best['f1']:.4f}  "
              f"@ proba-threshold={best['threshold']:.2f}  "
              f"(positives={int(y_full.sum())}/{len(y_full)} across train+val)")

    # Refit final probes on the FULL train+val pool, each with its own
    # calibrated C, now that (threshold, C) have been honestly selected via
    # cross-validation rather than one small, noisy held-out split.
    probes = {}
    for c in candidate_codes:
        y = np.array(Y_by_code[c])
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=chosen_C[c])
        clf.fit(X_pool, y)
        probes[c] = clf

    return probes, thresholds


# --------------------------------------------------------------------------
# 8b. Evidence Sentence Retrieval: BM25 + Cosine Similarity +
#     Classifier-Based Relevance -> Top-m Evidence Sentences.
#     This is the "Select Evidence Sentences for each predicted Statute" box.
#     Operates over `candidate_codes` / the frozen+whitened `bank_emb` --
#     independent of the probes' decision (which sections are predicted),
#     it only decides WHICH sentences best support a given prediction.
# --------------------------------------------------------------------------
def _bm25_tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def bm25_sentence_scores(sentences, query_text):
    """BM25 lexical relevance of each of THIS case's sentences against a
    statute's title+description. The BM25 index is built per-case (over
    just that case's own sentences) so the score reflects which of the
    case's sentences best match the statute, not corpus-wide term rarity."""
    if not sentences:
        return np.zeros(0)
    bm25 = BM25Okapi([_bm25_tokenize(s) for s in sentences])
    scores = np.array(bm25.get_scores(_bm25_tokenize(query_text)), dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / span if span > 1e-9 else np.zeros_like(scores)


def _minmax(x):
    x = np.asarray(x, dtype=np.float64)
    span = x.max() - x.min()
    return (x - x.min()) / span if span > 1e-9 else np.zeros_like(x)


class RelevanceClassifier(nn.Module):
    """Classifier-Based Relevance head: P(sentence supports statute) from
    [sent_emb ; proto_emb ; sent_emb * proto_emb] using the same frozen,
    whitened InLegalBERT embeddings already computed for the similarity
    matrix. This classifier is a small trainable MLP -- the underlying
    encoder itself remains frozen, so this stays consistent with Run 2's
    "no contrastive fine-tuning" baseline."""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 3, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, 1),
        )

    def forward(self, sent_emb, proto_emb):
        feat = torch.cat([sent_emb, proto_emb, sent_emb * proto_emb], dim=-1)
        return self.net(feat).squeeze(-1)


def train_relevance_classifier():
    """Trains RelevanceClassifier on (sentence, statute) pairs from the same
    `explanation`-derived positives as positive_pairs_from_explanation(),
    restricted to `candidate_codes`, with `relevance_clf_neg_per_pos` sampled
    negative candidate codes per positive. Frozen, whitened embeddings from
    the encoder are used as features, so this trains fast (a few epochs of a
    small MLP) and never touches the InLegalBERT weights."""
    pos_pairs = []
    for rec in train_split:
        pos_pairs.extend(positive_pairs_from_explanation(
            rec["fact"], rec.get("explanation", {}) or {}, rec["gold_sections"]))
    pos_pairs = [(s, c) for s, c in pos_pairs if c in candidate_codes]
    if not pos_pairs:
        print("[relevance-classifier] no (sentence, statute) positives found -- "
              "evidence retrieval will fall back to BM25 + cosine only.")
        return None

    code_to_idx = {c: i for i, c in enumerate(candidate_codes)}
    uniq_sentences = list({s for s, _ in pos_pairs})
    sent_emb_lookup = {s: e for s, e in zip(uniq_sentences, embed_and_whiten(uniq_sentences, cfg.max_length))}

    X_sent, X_proto, y = [], [], []
    for s, c in pos_pairs:
        X_sent.append(sent_emb_lookup[s]); X_proto.append(bank_emb[code_to_idx[c]]); y.append(1.0)
        neg_pool = [cc for cc in candidate_codes if cc != c]
        for nc in random.sample(neg_pool, min(cfg.relevance_clf_neg_per_pos, len(neg_pool))):
            X_sent.append(sent_emb_lookup[s]); X_proto.append(bank_emb[code_to_idx[nc]]); y.append(0.0)

    X_sent = torch.stack(X_sent).to(DEVICE)
    X_proto = torch.stack(X_proto).to(DEVICE)
    y = torch.tensor(y, dtype=torch.float32, device=DEVICE)

    clf = RelevanceClassifier(X_sent.shape[-1], cfg.relevance_clf_hidden).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=cfg.relevance_clf_lr)

    clf.train()
    for epoch in range(1, cfg.relevance_clf_epochs + 1):
        idx = torch.randperm(len(y), device=DEVICE)
        total, steps = 0.0, 0
        for i in range(0, len(y), cfg.relevance_clf_batch_size):
            b = idx[i:i + cfg.relevance_clf_batch_size]
            opt.zero_grad()
            logits = clf(X_sent[b], X_proto[b])
            loss = F.binary_cross_entropy_with_logits(logits, y[b])
            loss.backward()
            opt.step()
            total += loss.item(); steps += 1
        print(f"  [relevance-classifier] epoch {epoch}/{cfg.relevance_clf_epochs} "
              f"BCE loss = {total / max(1, steps):.4f}")
    clf.eval()
    return clf


@torch.no_grad()
def classifier_relevance_scores(clf, sentences, proto_vec):
    if clf is None or not sentences:
        return np.zeros(len(sentences))
    sent_emb = embed_and_whiten(sentences, cfg.max_length).to(DEVICE)
    proto_batch = proto_vec.to(DEVICE).unsqueeze(0).expand(len(sentences), -1)
    probs = torch.sigmoid(clf(sent_emb, proto_batch)).cpu().numpy()
    return probs


def retrieve_evidence_sentences(sentences, section_code, relevance_clf, top_m=None):
    """Score S = {S1..Sn} via BM25 + Cosine Similarity + Classifier-Based
    Relevance, min-max normalise each signal, combine with the configured
    weights, and return the Top-m evidence sentences (sentence, combined_score)
    for `section_code`, ranked highest first."""
    top_m = top_m or cfg.top_m_evidence
    if not sentences:
        return []

    proto_idx = candidate_codes.index(section_code)
    proto_vec = bank_emb[proto_idx]

    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    cosine_raw = (sent_emb @ proto_vec.unsqueeze(-1)).squeeze(-1).numpy()
    cosine = _minmax(cosine_raw)

    query_text = f"{prototype_titles.get(section_code, '')} {prototype_texts.get(section_code, '')}".strip()
    bm25 = bm25_sentence_scores(sentences, query_text)

    clf_scores = _minmax(classifier_relevance_scores(relevance_clf, sentences, proto_vec))

    combined = (cfg.evidence_bm25_weight * bm25
                + cfg.evidence_cosine_weight * cosine
                + cfg.evidence_classifier_weight * clf_scores)

    order = np.argsort(-combined)[:min(top_m, len(sentences))]
    return [(sentences[i], float(combined[i])) for i in order]


def predict_case(fact_text, probes, thresholds, relevance_clf=None):
    feats, _legacy_evidence, _ = compute_features(fact_text)
    feat = feats.reshape(1, -1)
    probs = {c: float(probes[c].predict_proba(feat)[0, 1]) for c in candidate_codes}
    chosen = [(c, p) for c, p in probs.items() if p >= thresholds[c]]
    if not chosen:
        # No probe crossed its own threshold for this case -- fall back to the
        # single highest-confidence candidate rather than emitting zero labels.
        top_c = max(probs, key=probs.get)
        chosen = [(top_c, probs[top_c])]

    sentences = split_sentences(fact_text, cfg.max_sentences)
    results = []
    for c, p in chosen:
        evidence = retrieve_evidence_sentences(sentences, c, relevance_clf, cfg.top_m_evidence)
        results.append({
            "section": c,
            "score": round(float(p), 4),
            "evidence_sentences": [sent for sent, _ in evidence],
            "evidence_scores": [round(sc, 4) for _, sc in evidence],
        })
    return results


# --------------------------------------------------------------------------
# 9. LLM-Based Reasoning Generation: (IPC Section, Selected Evidence
#    Sentences, CoT prompting) -> Qwen -> Explanation.
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def build_cot_prompt(section_code, evidence_sentences, score):
    """Chain-of-Thought prompt: statute + ranked evidence -> (1) identify
    satisfied elements -> (2) connect each evidence sentence -> (3) final
    explanation. Matches Figure 1's "Input to LLM (IPC Section, Selected
    Evidence Sentences, CoT prompting)" box."""
    title = prototype_titles.get(section_code, "")
    statute_text = prototype_texts.get(section_code, "")
    evidence_block = "\n".join(f"  {i + 1}. {sent} (relevance={sc:.3f})"
                                for i, (sent, sc) in enumerate(evidence_sentences)) or "  (no evidence retrieved)"
    return (
        "You are a legal reasoning assistant analysing an Indian Penal Code (IPC) "
        "statute prediction. Think step by step before answering.\n\n"
        f"Candidate section: {section_code}{f' ({title})' if title else ''}\n"
        f"Statute text: {statute_text}\n"
        f"Probe confidence score: {score:.3f}\n\n"
        f"Evidence sentences retrieved from the case facts (ranked by combined "
        f"BM25 + cosine similarity + classifier relevance):\n{evidence_block}\n\n"
        "Step 1 - List which elements of the statute the evidence appears to satisfy.\n"
        "Step 2 - For each evidence sentence, state briefly how it supports (or "
        "weakens) applicability of this section.\n"
        "Step 3 - Give a final, concise legal explanation (2-4 sentences) linking "
        "the evidence to the statute.\n\n"
        "Respond with your Step 1 and Step 2 reasoning first, then a line "
        "'Final Explanation:' followed by the Step 3 explanation."
    )


def generate_explanation(section_code, evidence_sentences, score):
    """evidence_sentences: list of (sentence_text, combined_relevance_score) tuples,
    already ranked by retrieve_evidence_sentences(). Uses Qwen with CoT prompting
    when cfg.use_qwen_explainer=True; otherwise a deterministic template reproduces
    the same three-step structure without a decoder LLM."""
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        prompt = build_cot_prompt(section_code, evidence_sentences, score)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": prompt},
        ]
        chat_prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(chat_prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        full_output = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if "Final Explanation:" in full_output:
            return full_output.split("Final Explanation:", 1)[1].strip()
        return full_output

    # Deterministic fallback: same CoT structure, no decoder LLM required.
    title = prototype_titles.get(section_code, "")
    title_part = f" ({title})" if title else ""
    statute_text = prototype_texts.get(section_code, "")
    if not evidence_sentences:
        return (f"{section_code}{title_part} is predicted (probe confidence {score:.3f}), "
                f"but no evidence sentences were retrieved for this case.")
    top_sent, top_score = evidence_sentences[0]
    others = "; ".join(s[:100] for s, _ in evidence_sentences[1:])
    return (
        f"{section_code}{title_part} is predicted (probe confidence {score:.3f}). "
        f"Step 1: the case facts describe conduct matching the elements of \"{statute_text[:160]}...\". "
        f"Step 2: the strongest supporting sentence (relevance {top_score:.3f}) is \"{top_sent[:160]}\""
        + (f", further supported by: {others[:200]}." if others else ".")
        + " Step 3 (final explanation): the retrieved evidence, ranked by BM25 + embedding "
          "similarity + classifier relevance, is consistent with this statute applying to the case."
    )


# --------------------------------------------------------------------------
# 10. Evaluation
# --------------------------------------------------------------------------
def evaluate_run(run_name, probes, thresholds, relevance_clf=None, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], probes, thresholds, relevance_clf)
        for p in preds:
            evidence_pairs = list(zip(p["evidence_sentences"], p["evidence_scores"]))
            p["explanation"] = generate_explanation(p["section"], evidence_pairs, p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]
    labels = sorted({l for ls in gold + pred for l in ls})
    mlb = MultiLabelBinarizer(classes=labels)
    yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=labels, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


if __name__ == "__main__":
    # Run 2: prototype pipeline with the ORIGINAL (not fine-tuned) InLegalBERT
    # (law-ai/InLegalBERT), the full ipc_sections_clean.json prototype bank
    # (candidates restricted at inference time to the 7 codes with gold
    # supervision -- see section 5b), whitened embeddings, and a per-class
    # logistic-regression probe over the 7 prototype-similarity features,
    # each with its own calibrated decision threshold (see section 8) --
    # followed by multi-signal (BM25 + cosine + classifier) evidence
    # retrieval and CoT-prompted explanation generation (sections 8b, 9).
    print("\n[calibration] training + calibrating per-class probes:")
    probes, thresholds = train_and_calibrate_probes()

    print("\n[relevance classifier] training BM25+cosine+classifier evidence scorer "
          "on the frozen, whitened InLegalBERT embeddings ...")
    relevance_clf = train_relevance_classifier()

    run2_metrics, run2_predictions = evaluate_run(
        "run2_prototype_baseline", probes, thresholds, relevance_clf)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, topk_feature=3, probe_C=1.0, calib_n_splits=5, top_m_evidence=3, evidence_bm25_weight=0.3, evidence_cosine_weight=0.4, evidence_classifier_weight=0.3, relevance_clf_hidden=128, relevance_clf_epochs=5, relevance_clf_lr=0.001, relevance_clf_batch_size=64, relevance_clf_neg_per_pos=4, use_qwen_explainer=False, qwen_model_name='Qwen/Qwen2.5-1.5B-Instruct', qwen_max_new_tokens=220, out_dir='./prototype_contrastive_outputs')
525 cases | 574 statute prototypes
  IPC 1: This Act shall be called the Indian Penal Code, and shall extend to the whole of India except the State of Jam...
  IPC 10: The word “man” denotes a male human being of any age;
The word “woman” denotes a female h

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103
Observed labels in train+val: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506'] (7 of 574 catalog codes)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[whitening] fitting on prototype bank + training sentences ...
[whitening] fitted on 4574 vectors -> 256-dim whitened space
Restricting prediction candidates to 7 observed codes: ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

[calibration] training + calibrating per-class probes:


Featurizing:   0%|          | 0/422 [00:00<?, ?it/s]

  [IPC 147] best C=30  5-fold OOF F1=0.3795  @ proba-threshold=0.54  (positives=62/422 across train+val)
  [IPC 201] best C=30  5-fold OOF F1=0.2357  @ proba-threshold=0.47  (positives=41/422 across train+val)
  [IPC 302] best C=100  5-fold OOF F1=0.5732  @ proba-threshold=0.34  (positives=145/422 across train+val)
  [IPC 376] best C=30  5-fold OOF F1=0.4179  @ proba-threshold=0.56  (positives=66/422 across train+val)
  [IPC 420] best C=100  5-fold OOF F1=0.5538  @ proba-threshold=0.65  (positives=64/422 across train+val)
  [IPC 498A] best C=100  5-fold OOF F1=0.6338  @ proba-threshold=0.66  (positives=67/422 across train+val)
  [IPC 506] best C=100  5-fold OOF F1=0.2597  @ proba-threshold=0.55  (positives=52/422 across train+val)

[relevance classifier] training BM25+cosine+classifier evidence scorer on the frozen, whitened InLegalBERT embeddings ...
  [relevance-classifier] epoch 1/5 BCE loss = 0.4857
  [relevance-classifier] epoch 2/5 BCE loss = 0.4268
  [relevance-classifier] epoch

[run2_prototype_baseline] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run2_prototype_baseline.jsonl

--- run2_prototype_baseline: TEST metrics ---
              macro_f1: 0.4143
              micro_f1: 0.4092
           weighted_f1: 0.4482
       macro_precision: 0.3402
          macro_recall: 0.6044
       micro_precision: 0.2996
          micro_recall: 0.6452
  exact_match_accuracy: 0.0777
          hamming_loss: 0.3204
